# DAAT — PHASE 2 TEST APPLY (per-sequence track_thresh)
# Loads the FROZEN density->track_thresh rule from Phase 1 (perseq_rule.json),
# measures each TEST sequence's density from detector output (NO GT), looks up
# its bucket's track_thresh, and runs. Writes flat MOT .txt for submission.
# LEGITIMACY: rule fit on TRAIN only; no test GT or test score is consulted.


In [ ]:
!pip install -q gdown
!gdown --id 1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5 -O bytetrack_x_mot17.pth.tar

In [ ]:
# ============================================================================
# CELL 1: Mount Drive + install deps
# ============================================================================
# Minimal install. We need:
#   - numpy, scipy   (Hungarian assignment, IoU math) — already in Colab
#   - opencv-python  (read frames, draw boxes, write MP4) — already in Colab
#   - pandas         (parse det.txt / gt.txt as CSV) — already in Colab
#   - motmetrics     (MOTA/IDF1 evaluation) — NOT in Colab by default
#   - tqdm           (progress bars) — already in Colab
# YOLOX (ByteTrack MOT17+CrowdHuman weights) is the private detector (replaces det.txt).

!pip install torchreid -q
!pip install gdown -q
!pip install scipy -q
# YOLOX (ByteTrack pedestrian detector). --no-deps so it does NOT downgrade the
# Colab torch/torchvision/numpy that torchreid and the tracker depend on.
!pip install yolox loguru thop tabulate -q --no-deps

# Fetch the ByteTrack YOLOX-X weights (bytetrack_x_mot17.pth.tar) if needed,
# then set YOLOX_WEIGHTS in the config cell to the saved path:
#   !pip install -q gdown
#   !gdown --id 1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5 -O bytetrack_x_mot17.pth.tar

import sys, subprocess

def _pip_install(pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import motmetrics
except ImportError:
    print('Installing motmetrics ...')
    _pip_install(['motmetrics==1.4.0'])
    import motmetrics

# Set DATA_ROOT / OUTPUT_ROOT for your environment (Colab: mount Drive first)
import os
if not os.path.ismount('${DRIVE_ROOT}'):
    import os
DATA_ROOT = os.environ.get('DATA_ROOT', '/content/data')
OUTPUT_ROOT = os.environ.get('OUTPUT_ROOT', '/content/outputs')

print('motmetrics version:', motmetrics.__version__)
print('Drive mounted at ${DRIVE_ROOT}')

In [ ]:
import os

MOT17_TRAIN_DIR = '${DATA_ROOT}/MOT17/test'  # TEST set (no GT)

OUTPUT_DIR = ('${PROJECT_ROOT}/2026/'
              'MOT/MOT-17/TEST/SUBMISSION-Test14-perseq/results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- All 21 sequences -------------------------------------------------------
BASE_SEQS = ['MOT17-01', 'MOT17-03', 'MOT17-06',
             'MOT17-07', 'MOT17-08', 'MOT17-12', 'MOT17-14']  # TEST sequences
DETECTORS = ['DPM', 'FRCNN', 'SDP']
SEQUENCES = [f'{b}-{d}' for b in BASE_SEQS for d in DETECTORS]   # all 21

SKIP_IF_DONE = True
RENDER_MP4   = False
COMBINED_CSV = os.path.join(OUTPUT_DIR, 'ALL_SEQUENCES_metrics.csv')

# Core ByteTrack hyperparameters -- unchanged
TRACK_THRESH        = 0.5    # default; OVERRIDDEN per-sequence by frozen rule
# --- Phase-2: frozen rule from Phase-1 calibration ---------------------------
# Phase-1 wrote the rule into its OUTPUT_ROOT (the 'ALL-21-stage1-fusion-EASY' folder).
RULE_JSON = ('${PROJECT_ROOT}/2026/'
             'MOT/MOT-17/TEST/Test-8-YOLOX/ALL-21-stage1-fusion-EASY/perseq_rule.json')
# Robust fallback: if the exact path is wrong, search the Shared Drive for it.
if not os.path.exists(RULE_JSON):
    import glob as _g
    _hits = _g.glob('${PROJECT_ROOT}/**/perseq_rule.json',
                    recursive=True)
    assert _hits, ('perseq_rule.json not found anywhere under the Shared Drive. '
                   'Run Phase 1 first, or set RULE_JSON manually.')
    RULE_JSON = sorted(_hits, key=os.path.getmtime)[-1]   # most recent
    print('RULE_JSON auto-located ->', RULE_JSON)
PERSEQ_DENSITY_CACHE = os.path.join(os.path.dirname(OUTPUT_DIR), 'test_density.json')
MATCH_THRESH        = 0.8
TRACK_BUFFER        = 30
MIN_BOX_AREA        = 100
ASPECT_RATIO_THRESH = 1.6
MOT20_FLAG          = False

# ============================================================
# Test-6 configuration
# (inherits Test-5 gate values as starting defaults;
#  adaptive gates will override per-sequence at warmup)
# ============================================================
REID_GLOBAL_FLOOR = 0.65   # unchanged
REID_MARGIN       = 0.05   # Test-5 value (adaptive override will adjust)
REID_THETA_CAP    = 0.80   # Test-5 value (adaptive override will adjust)
REID_DEVICE       = 'cuda'

# D_low: ENABLED — ByteTrack low-confidence second association
# (motion/IoU rescue of lost tracks using detections in [0.1, TRACK_THRESH]).
USE_DLOW_RESCUE   = False
DLOW_IOU_THRESH   = 0.5

# ---- Occlusion-aware re-ID -------------------------------------------------
# When True: SUSPEND a track during occlusion (skip the blind D_low IoU rescue)
# and re-identify on reappearance using BOTH appearance correlation AND motion
# consistency: revive the same ID only if the reappearing detection both looks
# like the lost track (registry) AND lands near its motion-predicted position;
# otherwise mint a new ID. Tolerance grows with how long the track was occluded.
USE_OCCLUSION_REID     = False
OCC_MOTION_GATE_BASE   = 1.5   # base gate radius, in track-diagonal units
OCC_MOTION_GATE_GROWTH = 0.10  # gate grows this many diagonals per lost frame

# ---- NEW: Camera Motion Compensation (GMC) ----------------------------------
# Estimates affine transform between frames using sparse LK optical flow.
# Corrects Kalman-predicted boxes before IoU matching.
# Targets the SDP-sequence gap where camera movement caused drift.
USE_GMC = True

# ---- NEW: Scene-Adaptive Gates ----------------------------------------------
# At frame ADAPTIVE_WARMUP_FRAMES, computes mean pairwise cosine similarity
# between mature registry templates to estimate scene difficulty, then
# interpolates margin + theta_cap between easy and hard endpoints.
ADAPTIVE_GATES         = True
ADAPTIVE_WARMUP_FRAMES = 30     # number of frames before difficulty is assessed
ADAPTIVE_MARGIN_EASY   = 0.07   # margin for easy scenes (sim < 0.45, distinct people)
ADAPTIVE_MARGIN_HARD   = 0.03   # margin for hard scenes (sim > 0.60, similar people)
ADAPTIVE_CAP_EASY      = 0.85   # theta cap for easy scenes
ADAPTIVE_CAP_HARD      = 0.75   # theta cap for hard scenes

# ---- NEW (Test-8): Stage-1 cost fusion ---------------------------------------
# BoT-SORT-style min(IoU, appearance) in the FIRST association. Appearance
# (frozen-median OSNet template, mature n>=5 only) can override the IoU cost
# only for spatially plausible pairs (IoU cost <= FUSION_PROXIMITY_THRESH,
# i.e. IoU >= 0.5 -- exactly the crossing regime where switches occur) and
# only when cosine similarity clears the appearance floor. The floor is
# scene-adaptive when FUSION_ADAPTIVE=True (ablation: set False for fixed).
USE_STAGE1_FUSION       = True
FUSION_PROXIMITY_THRESH = 0.5    # appearance trusted only where IoU cost <= 0.5
FUSION_SIM_FLOOR        = 0.60   # fixed floor (used when FUSION_ADAPTIVE=False)
FUSION_ADAPTIVE         = True   # interpolate floor from measured scene difficulty
FUSION_FLOOR_EASY       = 0.55   # easy scene (distinct people)  -> permissive
FUSION_FLOOR_HARD       = 0.70   # hard scene (look-alike crowd) -> strict

# ---- Private detection: YOLOX-X (ByteTrack pedestrian weights) --------------
# Community weights trained on MOT17 + CrowdHuman (bytetrack_x_mot17.pth.tar).
# Download from the ByteTrack release and point YOLOX_WEIGHTS at the .pth.tar.
USE_YOLO       = True            # True = use private detector (now YOLOX)
YOLOX_WEIGHTS  = '/content/bytetrack_x_mot17.pth.tar'
YOLOX_DEPTH    = 1.33            # YOLOX-X depth
YOLOX_WIDTH    = 1.25            # YOLOX-X width
YOLOX_NUM_CLS  = 1               # ByteTrack weights are single-class (person)
YOLOX_TSIZE    = (800, 1440)     # ByteTrack test size (H, W)
YOLOX_NMS      = 0.7             # NMS IoU threshold
YOLOX_CONF     = 0.1             # keep detections with score >= this for tracking
YOLOX_PP_CONF  = 0.001           # postprocess prefilter (matches ByteTrack test_conf)
YOLOX_RGB_MEAN = (0.485, 0.456, 0.406)   # ImageNet mean — ByteTrack preprocessing
YOLOX_RGB_STD  = (0.229, 0.224, 0.225)   # ImageNet std  — ByteTrack preprocessing
YOLO_BATCH     = 8               # frames per GPU batch (name kept for run_tracking)

print("Row-2 config: full tracker + YOLOX-X detector | ALL 21 MOT17 sequences")
print(f"  use_gmc              = {USE_GMC}")
print(f"  adaptive_gates       = {ADAPTIVE_GATES}")
print(f"  warmup_frames        = {ADAPTIVE_WARMUP_FRAMES}")
print(f"  margin easy/hard     = {ADAPTIVE_MARGIN_EASY} / {ADAPTIVE_MARGIN_HARD}")
print(f"  theta_cap easy/hard  = {ADAPTIVE_CAP_EASY} / {ADAPTIVE_CAP_HARD}")
print(f"  reid_global_floor    = {REID_GLOBAL_FLOOR}")
print(f"  use_dlow_rescue      = {USE_DLOW_RESCUE}")
print(f"  use_occlusion_reid   = {USE_OCCLUSION_REID}")
print(f"  use_stage1_fusion    = {USE_STAGE1_FUSION}")
print(f"  fusion floors        = fixed={FUSION_SIM_FLOOR} adaptive={FUSION_ADAPTIVE} easy/hard={FUSION_FLOOR_EASY}/{FUSION_FLOOR_HARD}")
print(f"  fusion proximity     = {FUSION_PROXIMITY_THRESH} (IoU cost ceiling for appearance)")
print(f"  output               = {OUTPUT_DIR}")
print(f"  sequences            = {len(SEQUENCES)}")


In [ ]:
# =============================================================================
# CELL 4 (v1-full-debug) -- Full IDS Diagnostic with Feature Vectors,
#                            IoU Matrix, Scores, and Bounding Boxes
# =============================================================================
# Every IDS-related event is recorded with:
#   frame_id        : frame number
#   stage           : which stage (3, 5, 6_revive, 6_new)
#   old_id          : previous track_id (or -1 if none)
#   new_id          : assigned track_id
#   reason          : textual description
#   det_score       : YOLO detection confidence score
#   det_box         : detection bounding box [x1,y1,x2,y2]
#   track_box       : Kalman-predicted track box (if available)
#   iou_cost        : IoU cost between det and matched/attempted track
#   cosine_sim      : cosine similarity vs registry template (if queried)
#   cosine_theta    : acceptance threshold applied
#   cosine_margin   : sim - second_best_sim
#   feat_norm       : L2 norm of the raw feature (should be ~1.0)
#   feat_mean_sim   : similarity between this feature and registry mean
#   n_registry      : number of mature templates in registry at this frame
#   n_lost          : number of lost tracks at this frame
#   n_template_enrolments : how many enrolments this track_id has
#
# Additionally, for frames where ANY IDS event occurs, the full IoU cost
# matrix is saved to a CSV so you can inspect every pairing that frame.
#
# All data saved to:
#   <OUTPUT_DIR>/ids_events.csv       -- one row per IDS event
#   <OUTPUT_DIR>/iou_matrices/        -- one CSV per IDS frame
#   tracker.print_ids_summary()       -- printed to console
# =============================================================================

import numpy as np
import scipy.linalg
from scipy.optimize import linear_sum_assignment
from collections import OrderedDict, defaultdict
import base64, csv, os, io

import torch
import torch.nn.functional as F
import torchvision.transforms as T
import torchreid
import cv2


class KalmanFilter:
    def __init__(self):
        ndim, dt = 4, 1.
        self._motion_mat = np.eye(2 * ndim, 2 * ndim)
        for i in range(ndim):
            self._motion_mat[i, ndim + i] = dt
        self._update_mat = np.eye(ndim, 2 * ndim)
        self._std_weight_position = 1. / 20
        self._std_weight_velocity = 1. / 160

    def initiate(self, measurement):
        mean_pos = measurement
        mean_vel = np.zeros_like(mean_pos)
        mean = np.r_[mean_pos, mean_vel]
        std = [2*self._std_weight_position*measurement[3],
               2*self._std_weight_position*measurement[3], 1e-2,
               2*self._std_weight_position*measurement[3],
               10*self._std_weight_velocity*measurement[3],
               10*self._std_weight_velocity*measurement[3], 1e-5,
               10*self._std_weight_velocity*measurement[3]]
        return mean, np.diag(np.square(std))

    def predict(self, mean, covariance):
        std_pos = [self._std_weight_position*mean[3],
                   self._std_weight_position*mean[3], 1e-2,
                   self._std_weight_position*mean[3]]
        std_vel = [self._std_weight_velocity*mean[3],
                   self._std_weight_velocity*mean[3], 1e-5,
                   self._std_weight_velocity*mean[3]]
        motion_cov = np.diag(np.square(np.r_[std_pos, std_vel]))
        mean = np.dot(self._motion_mat, mean)
        covariance = np.linalg.multi_dot((self._motion_mat, covariance, self._motion_mat.T)) + motion_cov
        return mean, covariance

    def multi_predict(self, means, covariances):
        std_pos = [self._std_weight_position*means[:,3],
                   self._std_weight_position*means[:,3],
                   1e-2*np.ones_like(means[:,3]),
                   self._std_weight_position*means[:,3]]
        std_vel = [self._std_weight_velocity*means[:,3],
                   self._std_weight_velocity*means[:,3],
                   1e-5*np.ones_like(means[:,3]),
                   self._std_weight_velocity*means[:,3]]
        sqr = np.square(np.r_[std_pos, std_vel]).T
        motion_cov = np.array([np.diag(sqr[i]) for i in range(len(means))])
        means = np.dot(means, self._motion_mat.T)
        left = np.dot(self._motion_mat, covariances).transpose((1,0,2))
        covariances = np.dot(left, self._motion_mat.T) + motion_cov
        return means, covariances

    def update(self, mean, covariance, measurement):
        std = [self._std_weight_position*mean[3],
               self._std_weight_position*mean[3], 1e-1,
               self._std_weight_position*mean[3]]
        innovation_cov = np.diag(np.square(std))
        projected_mean = np.dot(self._update_mat, mean)
        projected_cov = np.linalg.multi_dot((self._update_mat, covariance, self._update_mat.T)) + innovation_cov
        chol_factor, lower = scipy.linalg.cho_factor(projected_cov, lower=True, check_finite=False)
        kalman_gain = scipy.linalg.cho_solve(
            (chol_factor, lower), np.dot(covariance, self._update_mat.T).T, check_finite=False).T
        innovation = measurement - projected_mean
        new_mean = mean + np.dot(innovation, kalman_gain.T)
        new_covariance = covariance - np.linalg.multi_dot((kalman_gain, projected_cov, kalman_gain.T))
        return new_mean, new_covariance


class TrackState:
    New=0; Tracked=1; Lost=2; Removed=3


class STrack:
    shared_kalman = KalmanFilter()
    _next_id_counter = 0

    def __init__(self, tlwh, score):
        self._tlwh = np.asarray(tlwh, dtype=float)
        self.kalman_filter = None
        self.mean, self.covariance = None, None
        self.is_activated = False
        self.score = score
        self.tracklet_len = 0
        self.state = TrackState.New
        self.track_id = 0
        self.frame_id = 0
        self.start_frame = 0
        self.time_since_update = 0
        self.template_mean = None
        self.template_n = 0
        self.template_s_min = 1.0
        self.template_s_max = -1.0

    @staticmethod
    def next_id():
        STrack._next_id_counter += 1
        return STrack._next_id_counter

    @staticmethod
    def reset_id_counter():
        STrack._next_id_counter = 0

    @staticmethod
    def tlwh_to_xyah(tlwh):
        ret = np.asarray(tlwh).copy()
        ret[:2] += ret[2:] / 2
        ret[2] /= ret[3]
        return ret

    @staticmethod
    def tlbr_to_tlwh(tlbr):
        ret = np.asarray(tlbr).copy()
        ret[2:] -= ret[:2]
        return ret

    @property
    def tlwh(self):
        if self.mean is None:
            return self._tlwh.copy()
        ret = self.mean[:4].copy()
        ret[2] *= ret[3]
        ret[:2] -= ret[2:] / 2
        return ret

    @property
    def tlbr(self):
        ret = self.tlwh.copy()
        ret[2:] += ret[:2]
        return ret

    def activate(self, kalman_filter, frame_id, force_id=None):
        self.kalman_filter = kalman_filter
        self.track_id = force_id if force_id is not None else STrack.next_id()
        self.mean, self.covariance = self.kalman_filter.initiate(self.tlwh_to_xyah(self._tlwh))
        self.tracklet_len = 0
        self.state = TrackState.Tracked
        if frame_id == 1: self.is_activated = True
        self.frame_id = frame_id
        self.start_frame = frame_id
        self.time_since_update = 0

    def re_activate(self, new_track, frame_id, new_id=False):
        self.mean, self.covariance = self.kalman_filter.update(
            self.mean, self.covariance, self.tlwh_to_xyah(new_track.tlwh))
        self.tracklet_len = 0
        self.state = TrackState.Tracked
        self.is_activated = True
        self.frame_id = frame_id
        if new_id: self.track_id = STrack.next_id()
        self.score = new_track.score
        self.time_since_update = 0

    def update(self, new_track, frame_id):
        self.frame_id = frame_id
        self.tracklet_len += 1
        self.mean, self.covariance = self.kalman_filter.update(
            self.mean, self.covariance, self.tlwh_to_xyah(new_track.tlwh))
        self.state = TrackState.Tracked
        self.is_activated = True
        self.score = new_track.score
        self.time_since_update = 0

    def predict(self):
        mean_state = self.mean.copy()
        if self.state != TrackState.Tracked: mean_state[7] = 0
        self.mean, self.covariance = self.kalman_filter.predict(mean_state, self.covariance)
        self.time_since_update += 1

    @staticmethod
    def multi_predict(stracks):
        if len(stracks) > 0:
            multi_mean = np.asarray([st.mean.copy() for st in stracks])
            multi_covariance = np.asarray([st.covariance for st in stracks])
            for i, st in enumerate(stracks):
                if st.state != TrackState.Tracked: multi_mean[i][7] = 0
            multi_mean, multi_covariance = STrack.shared_kalman.multi_predict(multi_mean, multi_covariance)
            for i, (m, c) in enumerate(zip(multi_mean, multi_covariance)):
                stracks[i].mean = m
                stracks[i].covariance = c
                stracks[i].time_since_update += 1

    def mark_lost(self):    self.state = TrackState.Lost
    def mark_removed(self): self.state = TrackState.Removed
    def end_frame(self):    return self.frame_id


def ious(atlbrs, btlbrs):
    ious_arr = np.zeros((len(atlbrs), len(btlbrs)), dtype=float)
    if ious_arr.size == 0: return ious_arr
    a = np.asarray(atlbrs, dtype=float)
    b = np.asarray(btlbrs, dtype=float)
    aa = (a[:,2]-a[:,0])*(a[:,3]-a[:,1])
    bb = (b[:,2]-b[:,0])*(b[:,3]-b[:,1])
    for i in range(len(a)):
        xx1=np.maximum(a[i,0],b[:,0]); yy1=np.maximum(a[i,1],b[:,1])
        xx2=np.minimum(a[i,2],b[:,2]); yy2=np.minimum(a[i,3],b[:,3])
        w=np.maximum(0.,xx2-xx1); h=np.maximum(0.,yy2-yy1)
        inter=w*h
        ious_arr[i]=inter/(aa[i]+bb-inter+1e-9)
    return ious_arr

def iou_distance(atracks, btracks):
    if (len(atracks)>0 and isinstance(atracks[0],np.ndarray)) or        (len(btracks)>0 and isinstance(btracks[0],np.ndarray)):
        atlbrs,btlbrs=atracks,btracks
    else:
        atlbrs=[t.tlbr for t in atracks]
        btlbrs=[t.tlbr for t in btracks]
    return 1-ious(atlbrs,btlbrs)

def fuse_score(cost_matrix, detections):
    if cost_matrix.size==0: return cost_matrix
    iou_sim = 1-cost_matrix
    det_scores = np.array([d.score for d in detections])
    det_scores = np.expand_dims(det_scores,axis=0).repeat(cost_matrix.shape[0],axis=0)
    return 1-iou_sim*det_scores

def linear_assignment(cost_matrix, thresh):
    if cost_matrix.size==0:
        return (np.empty((0,2),dtype=int),
                tuple(range(cost_matrix.shape[0])),
                tuple(range(cost_matrix.shape[1])))
    cost_matrix=np.where(cost_matrix>thresh, thresh+1e-4, cost_matrix)
    row_ind,col_ind=linear_sum_assignment(cost_matrix)
    matches=[]
    for r,c in zip(row_ind,col_ind):
        if cost_matrix[r,c]<=thresh: matches.append([r,c])
    matches=np.asarray(matches,dtype=int) if matches else np.empty((0,2),dtype=int)
    ua=tuple(set(range(cost_matrix.shape[0]))-set(matches[:,0])) if matches.size else tuple(range(cost_matrix.shape[0]))
    ub=tuple(set(range(cost_matrix.shape[1]))-set(matches[:,1])) if matches.size else tuple(range(cost_matrix.shape[1]))
    return matches,ua,ub

def joint_stracks(tlista,tlistb):
    exists={}; res=[]
    for t in tlista: exists[t.track_id]=1; res.append(t)
    for t in tlistb:
        if exists.get(t.track_id,0)==0: exists[t.track_id]=1; res.append(t)
    return res

def sub_stracks(tlista,tlistb):
    stracks={t.track_id:t for t in tlista}
    for t in tlistb:
        if t.track_id in stracks: del stracks[t.track_id]
    return list(stracks.values())

def remove_duplicate_stracks(stracksa,stracksb):
    pdist=iou_distance(stracksa,stracksb)
    pairs=np.where(pdist<0.15)
    dupa,dupb=[],[]
    for p,q in zip(*pairs):
        timep=stracksa[p].frame_id-stracksa[p].start_frame
        timeq=stracksb[q].frame_id-stracksb[q].start_frame
        if timep>timeq: dupb.append(q)
        else: dupa.append(p)
    resa=[t for i,t in enumerate(stracksa) if i not in dupa]
    resb=[t for i,t in enumerate(stracksb) if i not in dupb]
    return resa,resb


class ReIDExtractor:
    def __init__(self, device='cuda', model_name='osnet_x1_0'):
        self.device = torch.device(device)
        self.model = torchreid.models.build_model(
            name=model_name, num_classes=1000, loss='softmax', pretrained=True)
        msmt17_urls = [
            'https://drive.google.com/uc?id=112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M',
            'https://drive.google.com/uc?id=1IosIFlLiulGIjwW3H8uMRmx3MzPwf86x',
        ]
        loaded_msmt = False
        for url in msmt17_urls:
            try:
                import gdown
                cache_dir = os.path.expanduser('~/.cache/torch/checkpoints')
                os.makedirs(cache_dir, exist_ok=True)
                ckpt_path = os.path.join(cache_dir, 'osnet_x1_0_msmt17.pth')
                if not os.path.exists(ckpt_path):
                    gdown.download(url, ckpt_path, quiet=False, fuzzy=True)
                if os.path.exists(ckpt_path) and os.path.getsize(ckpt_path) > 1_000_000:
                    torchreid.utils.load_pretrained_weights(self.model, ckpt_path)
                    print(f'   Loaded MSMT17 weights from {ckpt_path}')
                    loaded_msmt = True; break
            except Exception as e:
                print(f'   MSMT17 load failed: {e}'); continue
        if not loaded_msmt:
            print('   Using ImageNet-pretrained OSNet.')
        self.model = self.model.to(self.device)
        self.model.eval()
        self.transform = T.Compose([
            T.ToPILImage(), T.Resize((256,128)), T.ToTensor(),
            T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])

    @torch.no_grad()
    def extract(self, crops_bgr):
        if len(crops_bgr)==0: return np.zeros((0,512),dtype=np.float32)
        batch=[]
        for c in crops_bgr:
            if c is None or c.size==0 or c.shape[0]<2 or c.shape[1]<2:
                c=np.zeros((4,4,3),dtype=np.uint8)
            rgb=cv2.cvtColor(c,cv2.COLOR_BGR2RGB)
            batch.append(self.transform(rgb))
        x=torch.stack(batch,dim=0).to(self.device)
        feats=self.model(x)
        if isinstance(feats,(tuple,list)): feats=feats[0]
        feats=F.normalize(feats,p=2,dim=1)
        return feats.cpu().numpy().astype(np.float32)


def crop_from_image(img, tlbr):
    H,W=img.shape[:2]
    x1,y1,x2,y2=tlbr
    x1=int(max(0,min(W-1,np.floor(x1)))); y1=int(max(0,min(H-1,np.floor(y1))))
    x2=int(max(0,min(W,np.ceil(x2))));    y2=int(max(0,min(H,np.ceil(y2))))
    if x2-x1<2 or y2-y1<2: return None
    return img[y1:y2,x1:x2].copy()


class TemplateRegistry:
    def __init__(self):
        self.entries = OrderedDict()

    def update(self, track_id, feat_512_l2, frame_id):
        e=self.entries.get(track_id)
        if e is None:
            self.entries[track_id]={'mean':feat_512_l2.astype(np.float32).copy(),
                                    'n':1,'s_min':1.0,'s_max':1.0,'last_seen':frame_id}
            return
        e['n']+=1
        e['mean']+=( feat_512_l2-e['mean'])/e['n']
        nrm=np.linalg.norm(e['mean'])
        if nrm>1e-8: e['mean']/=nrm
        s=float(np.dot(feat_512_l2,e['mean']))
        e['s_min']=min(e['s_min'],s); e['s_max']=max(e['s_max'],s)
        e['last_seen']=frame_id

    MIN_TEMPLATE_N_FOR_QUERY=5

    def query(self, feat_512_l2):
        mature_ids=[i for i,e in self.entries.items() if e['n']>=self.MIN_TEMPLATE_N_FOR_QUERY]
        if not mature_ids: return None,None,None,{}
        M=np.stack([self.entries[i]['mean'] for i in mature_ids],axis=0)
        sims=M@feat_512_l2
        order=np.argsort(-sims)
        best_id=mature_ids[order[0]]
        best_sim=float(sims[order[0]])
        second_sim=float(sims[order[1]]) if len(order)>1 else -1.0
        # Return full similarity map for diagnostics
        sim_map={mature_ids[i]: float(sims[i]) for i in range(len(mature_ids))}
        return best_id,best_sim,second_sim,sim_map

    def acceptance_threshold(self, track_id, global_floor=0.5, theta_cap=None):
        # Rec 2: theta grows with template self-consistency (s_min). Capping it
        # stops a very self-consistent template from raising the acceptance bar
        # so high that valid revivals of the SAME person get rejected.
        theta = max(self.entries[track_id]['s_min'], global_floor)
        if theta_cap is not None:
            theta = min(theta, theta_cap)
        return theta

    def get_n_enrolments(self, track_id):
        e=self.entries.get(track_id)
        return e['n'] if e else 0

    @staticmethod
    def _encode(arr):
        return base64.b64encode(arr.astype(np.float16).tobytes()).decode('ascii')
    @staticmethod
    def _decode(s):
        return np.frombuffer(base64.b64decode(s.encode('ascii')),dtype=np.float16).astype(np.float32)


    def compute_scene_difficulty(self, n_warmup=5):
        """
        Estimate scene difficulty as the mean pairwise cosine similarity
        between all mature template means. High similarity = people look alike
        (crowd scene, hard). Low similarity = people look distinct (easy).
        Returns a float in [0, 1], or None if not enough templates.
        Uses at most 30 templates to keep computation fast (O(n^2)).
        """
        mature = [e for e in self.entries.values()
                  if e['n'] >= self.MIN_TEMPLATE_N_FOR_QUERY]
        if len(mature) < n_warmup:
            return None
        means = np.stack([e['mean'] for e in mature[:30]], axis=0)  # (k, 512)
        # Cosine similarity matrix (already L2-normalised)
        sim_matrix = means @ means.T                                # (k, k)
        k = len(means)
        # Mean of off-diagonal elements
        mask = ~np.eye(k, dtype=bool)
        mean_sim = float(sim_matrix[mask].mean())
        return mean_sim   # high -> hard scene; low -> easy scene

    def save_csv(self, path):
        with open(path,'w',newline='') as f:
            w=csv.writer(f)
            w.writerow(['track_id','n_clean_frames','template_mean_b64','s_min','s_max','last_seen_frame'])
            for tid,e in self.entries.items():
                w.writerow([tid,e['n'],self._encode(e['mean']),
                            f"{e['s_min']:.6f}",f"{e['s_max']:.6f}",e['last_seen']])

    def load_csv(self, path):
        if not os.path.exists(path): return
        with open(path,'r') as f:
            for row in csv.DictReader(f):
                tid=int(row['track_id'])
                self.entries[tid]={'mean':self._decode(row['template_mean_b64']),
                    'n':int(row['n_clean_frames']),'s_min':float(row['s_min']),
                    's_max':float(row['s_max']),'last_seen':int(row['last_seen_frame'])}


# =============================================================================
# BYTETracker v1 + FULL DIAGNOSTIC LOGGING
# =============================================================================

# =============================================================================
# GMC — Global Motion Compensation (BoT-SORT style, affine 2D)
# =============================================================================
# Estimates the affine transform between prev_frame and curr_frame using
# sparse Lucas-Kanade optical flow on GFTT feature points. Returns a 2x3
# affine matrix (or None if estimation fails / not enough points).
# The affine transform is applied to all Kalman-predicted track boxes so
# that the predicted positions are camera-motion-corrected before IoU matching.
# This directly addresses the SDP-sequence gap where camera movement caused
# Kalman predictions to drift away from detections even though IoU was strong.
# ===========================================================================

def estimate_affine_gmc(prev_gray, curr_gray,
                        max_corners=200, quality=0.01,
                        min_distance=3, block_size=3,
                        ransac_thresh=3.0):
    """
    Estimate 2D affine motion (scale + rotation + translation) between two
    grayscale frames using GFTT + LK optical flow + RANSAC.
    Returns (2x3 ndarray) or None if estimation fails.
    """
    if prev_gray is None or curr_gray is None:
        return None
    # Detect keypoints in previous frame
    pts = cv2.goodFeaturesToTrack(
        prev_gray, maxCorners=max_corners, qualityLevel=quality,
        minDistance=min_distance, blockSize=block_size)
    if pts is None or len(pts) < 4:
        return None
    # Track into current frame
    pts_curr, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray, curr_gray, pts, None,
        winSize=(21, 21), maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    if pts_curr is None:
        return None
    good_prev = pts[status.ravel() == 1]
    good_curr = pts_curr[status.ravel() == 1]
    if len(good_prev) < 4:
        return None
    # Estimate affine (partial: scale + rotation + translation, 4 DOF)
    M, inliers = cv2.estimateAffinePartial2D(
        good_prev, good_curr,
        method=cv2.RANSAC, ransacReprojThreshold=ransac_thresh)
    if M is None or (inliers is not None and inliers.sum() < 4):
        return None
    return M   # 2x3 float32


def apply_gmc_to_stracks(stracks, M):
    """
    Apply a 2x3 affine matrix M to the tlbr bounding boxes of all stracks
    in-place (via their Kalman mean state). Only moves the centre position;
    size is preserved.
    Stracks with mean=None (not yet activated) are skipped.
    """
    if M is None or len(stracks) == 0:
        return
    for t in stracks:
        if t.mean is None:
            continue
        # Extract centre (cx, cy) from the Kalman state (xyah format)
        cx, cy = float(t.mean[0]), float(t.mean[1])
        # Rotate/translate the centre
        pt = np.array([[[cx, cy]]], dtype=np.float32)
        pt_t = cv2.transform(pt, M)
        new_cx, new_cy = float(pt_t[0, 0, 0]), float(pt_t[0, 0, 1])
        t.mean[0] = new_cx
        t.mean[1] = new_cy

class BYTETracker:
    def __init__(self, track_thresh=0.5, match_thresh=0.8, track_buffer=30,
                 mot20=False, frame_rate=30, device='cuda',
                 reid_global_floor=0.65, reid_margin=0.10,
                 reid_theta_cap=None, use_dlow_rescue=False, dlow_iou_thresh=0.5,
                 # Test-6: Camera Motion Compensation
                 use_gmc=True,
                 # Test-6: Scene-Adaptive Gates
                 adaptive_gates=True,
                 adaptive_warmup_frames=30,
                 adaptive_margin_easy=0.07,   # margin when scene is easy (sim < 0.45)
                 adaptive_margin_hard=0.03,   # margin when scene is hard (sim > 0.60)
                 adaptive_cap_easy=0.85,      # theta cap when scene is easy
                 adaptive_cap_hard=0.75,      # theta cap when scene is hard
                 # Test-8: Stage-1 min(IoU, appearance) cost fusion (BoT-SORT-style)
                 use_stage1_fusion=False,
                 fusion_proximity_thresh=0.5,
                 fusion_sim_floor=0.60,
                 fusion_adaptive=True,
                 fusion_floor_easy=0.55,
                 fusion_floor_hard=0.70,
                 # Occlusion-aware re-ID
                 use_occlusion_reid=False,
                 occ_motion_gate_base=1.5,
                 occ_motion_gate_growth=0.10,
                 output_dir=None):
        self.tracked_stracks=[]; self.lost_stracks=[]; self.removed_stracks=[]
        self.frame_id=0
        self.track_thresh=track_thresh; self.match_thresh=match_thresh
        self.det_thresh=track_thresh+0.1
        self.buffer_size=int(frame_rate/30.0*track_buffer)
        self.max_time_lost=self.buffer_size
        self.kalman_filter=KalmanFilter(); self.mot20=mot20
        STrack.reset_id_counter()
        self.reid=ReIDExtractor(device=device)
        self.registry=TemplateRegistry()
        self.reid_global_floor=reid_global_floor
        self.reid_margin=reid_margin
        self.reid_theta_cap=reid_theta_cap
        self.use_dlow_rescue=use_dlow_rescue
        self.dlow_iou_thresh=dlow_iou_thresh
        # Occlusion-aware re-ID
        self.use_occlusion_reid=use_occlusion_reid
        self.occ_motion_gate_base=occ_motion_gate_base
        self.occ_motion_gate_growth=occ_motion_gate_growth
        # Test-8: Stage-1 fusion
        self.use_stage1_fusion=use_stage1_fusion
        self.fusion_proximity_thresh=fusion_proximity_thresh
        self.fusion_sim_floor=fusion_sim_floor
        self.fusion_adaptive=fusion_adaptive
        self.fusion_floor_easy=fusion_floor_easy
        self.fusion_floor_hard=fusion_floor_hard
        self._active_fusion_floor=fusion_sim_floor   # may adapt at warmup
        self._stage1_det_feats=None                  # per-frame det features (reused by Stage 3.5)
        self.fusion_stats={'frames_with_fusion':0,'cells_overridden':0,
                           'matches_decided_by_appearance':0,
                           'mature_template_rows_seen':0}
        # Test-6: GMC
        self.use_gmc=use_gmc
        self._prev_gray=None                         # previous frame (grayscale) for GMC
        self._gmc_matrix=None                        # last estimated affine matrix
        # Test-6: Adaptive gates
        self.adaptive_gates=adaptive_gates
        self.adaptive_warmup_frames=adaptive_warmup_frames
        self.adaptive_margin_easy=adaptive_margin_easy
        self.adaptive_margin_hard=adaptive_margin_hard
        self.adaptive_cap_easy=adaptive_cap_easy
        self.adaptive_cap_hard=adaptive_cap_hard
        self._scene_difficulty=None                  # computed at warmup
        self._active_margin=reid_margin              # effective margin (may adapt)
        self._active_theta_cap=reid_theta_cap        # effective cap (may adapt)
        self._scene_log=[]                           # log of difficulty estimates

        # ---- FULL DIAGNOSTIC LOG ----
        self.ids_log = []            # full rich event records
        self.ids_by_stage = defaultdict(int)
        self.iou_matrix_frames = {}  # frame_id -> {'tracks':[], 'dets':[], 'matrix':ndarray, 'fused':ndarray}
        self.output_dir = output_dir

        # ---- D_LOW DISCARD LOG ----
        # Tracks every detection that falls into D_low and is discarded.
        # For each D_low detection we also check whether it spatially
        # overlaps any track that gets marked lost that same frame
        # (IoU > 0.3). Those are the detections we could have used to
        # keep a track alive but threw away instead.
        self.dlow_log = []           # list of dicts, one per D_low detection per frame
        self.dlow_summary = {        # aggregate counters
            'total_dlow_dets': 0,
            'frames_with_dlow': 0,
            'dlow_overlapping_lost_track': 0,   # overlapped a lost track but NOT rescued
            'dlow_no_overlap': 0,               # genuine background
            'dlow_rescued': 0,                  # Rec 3: actually re-activated a lost track
        }

    def flush_registry(self, csv_path):
        self.registry.save_csv(csv_path)

    def load_registry(self, csv_path):
        self.registry.load_csv(csv_path)

    def _log_event(self, frame_id, stage, old_id, new_id, reason,
                   det_score=None, det_box=None, track_box=None,
                   iou_cost=None, fused_cost=None,
                   cosine_sim=None, cosine_theta=None, cosine_margin=None,
                   feat_norm=None, feat_mean_sim=None,
                   n_registry=None, n_lost=None, n_enrolments=None,
                   all_sims=None):
        """Record a full diagnostic event."""
        event = {
            'frame':            frame_id,
            'stage':            stage,
            'old_id':           old_id,
            'new_id':           new_id,
            'reason':           reason,
            'det_score':        round(det_score, 4) if det_score is not None else None,
            'det_box':          [round(v,1) for v in det_box] if det_box is not None else None,
            'track_box':        [round(v,1) for v in track_box] if track_box is not None else None,
            'iou_cost':         round(iou_cost, 4) if iou_cost is not None else None,
            'iou_overlap':      round(1-iou_cost, 4) if iou_cost is not None else None,
            'fused_cost':       round(fused_cost, 4) if fused_cost is not None else None,
            'cosine_sim':       round(cosine_sim, 4) if cosine_sim is not None else None,
            'cosine_theta':     round(cosine_theta, 4) if cosine_theta is not None else None,
            'cosine_margin':    round(cosine_margin, 4) if cosine_margin is not None else None,
            'feat_norm':        round(feat_norm, 4) if feat_norm is not None else None,
            'feat_mean_sim':    round(feat_mean_sim, 4) if feat_mean_sim is not None else None,
            'n_registry':       n_registry,
            'n_lost':           n_lost,
            'n_enrolments':     n_enrolments,
            'all_sims_top5':    str(sorted(all_sims.items(), key=lambda x:-x[1])[:5]) if all_sims else None,
        }
        self.ids_log.append(event)
        self.ids_by_stage[stage] += 1

    def _save_iou_matrix(self, frame_id, strack_pool, detections, raw_iou, fused):
        """Save the IoU matrix for a frame where IDS occurred."""
        self.iou_matrix_frames[frame_id] = {
            'track_ids':  [t.track_id for t in strack_pool],
            'track_boxes':[t.tlbr.tolist() for t in strack_pool],
            'track_states':[t.state for t in strack_pool],
            'det_scores': [d.score for d in detections],
            'det_boxes':  [d.tlbr.tolist() for d in detections],
            'iou_cost':   raw_iou.tolist(),
            'fused_cost': fused.tolist(),
        }

    def save_ids_csv(self, path):
        """Save all IDS events to a CSV file."""
        if not self.ids_log:
            print("   No IDS events logged.")
            return
        fieldnames = list(self.ids_log[0].keys())
        with open(path, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(self.ids_log)
        print(f"   Saved {len(self.ids_log)} IDS events -> {path}")

    def save_dlow_csv(self, path):
        """Save D_low discard log to CSV."""
        if not self.dlow_log:
            print("   No D_low detections logged.")
            return
        fieldnames = list(self.dlow_log[0].keys())
        with open(path, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(self.dlow_log)
        print(f"   Saved {len(self.dlow_log)} D_low events -> {path}")

    def print_dlow_summary(self):
        """Print D_low discard summary."""
        s = self.dlow_summary
        total = s['total_dlow_dets']
        rescued = s.get('dlow_rescued', 0)
        rescuable = s['dlow_overlapping_lost_track']
        genuine_bg = s['dlow_no_overlap']
        pct_resc_act = 100*rescued/total if total>0 else 0
        pct_rescue = 100*rescuable/total if total>0 else 0
        pct_bg     = 100*genuine_bg/total if total>0 else 0
        print()
        print("=" * 60)
        print("D_LOW DISCARD ANALYSIS")
        print("=" * 60)
        print(f"  Frames with D_low detections : {s['frames_with_dlow']}")
        print(f"  Total D_low detections       : {total}")
        print()
        print(f"  RESCUED (Rec 3, re-activated): {rescued:4d}  ({pct_resc_act:5.1f}%)")
        print(f"    -> matched a just-lost track at IoU>={self.dlow_iou_thresh:.2f}; id preserved")
        print()
        print(f"  Still overlap a lost track   : {rescuable:4d}  ({pct_rescue:5.1f}%)")
        print(f"    -> overlapped (IoU>=0.3) but below the {self.dlow_iou_thresh:.2f} rescue gate")
        print()
        print(f"  No overlap with any lost track: {genuine_bg:4d}  ({pct_bg:5.1f}%)")
        print(f"    -> genuine background / new arrivals / already tracked")
        print()
        # Score distribution of rescuable vs background
        rescuable_events = [e for e in self.dlow_log if e['could_rescue_lost_track']]
        bg_events        = [e for e in self.dlow_log if not e['could_rescue_lost_track']]
        if rescuable_events:
            sc = [e['det_score'] for e in rescuable_events]
            ov = [e['iou_overlap_with_lost'] for e in rescuable_events]
            import numpy as _np
            print(f"  Rescuable D_low score:  min={min(sc):.3f}  mean={_np.mean(sc):.3f}  max={max(sc):.3f}")
            print(f"  Rescuable D_low IoU:    min={min(ov):.3f}  mean={_np.mean(ov):.3f}  max={max(ov):.3f}")
            print()
        if bg_events:
            sc_bg = [e['det_score'] for e in bg_events]
            import numpy as _np
            print(f"  Background D_low score: min={min(sc_bg):.3f}  mean={_np.mean(sc_bg):.3f}  max={max(sc_bg):.3f}")
            print()
        print("First 20 D_low events (rescuable only):")
        print(f"  {'frame':>5}  {'score':>6}  {'iou_w_lost':>10}  {'lost_id':>8}  det_box")
        print("  " + "-"*70)
        shown = 0
        for e in self.dlow_log:
            if not e['could_rescue_lost_track']: continue
            print(f"  {e['frame']:>5}  {e['det_score']:>6.3f}  "
                  f"{e['iou_overlap_with_lost']:>10.3f}  "
                  f"{str(e['best_lost_track_id']):>8}  {e['det_box']}")
            shown += 1
            if shown >= 20: break
        print("=" * 60)
        print()

    def save_iou_matrices(self, out_dir):
        """Save per-frame IoU matrices to CSVs."""
        if not self.iou_matrix_frames:
            return
        mat_dir = os.path.join(out_dir, 'iou_matrices')
        os.makedirs(mat_dir, exist_ok=True)
        for frame_id, data in self.iou_matrix_frames.items():
            path = os.path.join(mat_dir, f'frame_{frame_id:05d}_iou.csv')
            track_ids = data['track_ids']
            det_scores = data['det_scores']
            raw = np.array(data['iou_cost'])
            fused = np.array(data['fused_cost'])
            with open(path, 'w', newline='') as f:
                w = csv.writer(f)
                # Header: track_id, state, track_box, then one col per detection
                det_headers = [f'det{j}_score={det_scores[j]:.3f}' for j in range(len(det_scores))]
                w.writerow(['track_id', 'track_state', 'track_box'] + det_headers)
                for i, tid in enumerate(track_ids):
                    state_name = ['New','Tracked','Lost','Removed'][data['track_states'][i]]
                    box_str = '['+','.join(f'{v:.1f}' for v in data['track_boxes'][i])+ ']'
                    iou_vals = [f'iou={1-raw[i,j]:.3f} fused={1-fused[i,j]:.3f}' for j in range(len(det_scores))]
                    w.writerow([tid, state_name, box_str] + iou_vals)
        print(f"   Saved IoU matrices for {len(self.iou_matrix_frames)} frames -> {mat_dir}/")

    def print_ids_summary(self):
        total = sum(self.ids_by_stage.values())
        print()
        print("=" * 70)
        print("IDS FULL DIAGNOSTIC SUMMARY")
        print("=" * 70)
        print(f"  Total events logged: {total}")
        print()

        # Rec 4: classify every event as a TRUE IDS vs an identity-preserving
        # re-activation. Only Stage 6b mints a new id while a lost track exists
        # -> that is the genuine ID-switch source. Stages 3, 5, 6a and the new
        # 4 (D_low rescue) all KEEP the original track_id.
        reactivation_stages = (3, 5, '6_revive', '4_dlow_rescue')
        n_reactivation = sum(self.ids_by_stage.get(k, 0) for k in reactivation_stages)
        n_true_ids     = self.ids_by_stage.get('6_new', 0)
        print("-" * 70)
        print("  IDS CLASSIFICATION (Rec 4):")
        print(f"    Identity-preserving re-activations : {n_reactivation:4d}"
              "  (Stage 3 + 5 + 6a + 4 D_low rescue; track_id unchanged)")
        print(f"    TRUE new-id mints (Stage 6b)       : {n_true_ids:4d}"
              "  <-- the genuine IDS source")
        print("-" * 70)
        print()

        stage_labels = {
            '4_dlow_rescue': "Stage 4  - D_low RESCUE (lost track kept alive, Rec 3)",
            3:          "Stage 3  - First assoc (lost track re-activated via IoU)",
            5:          "Stage 5  - Unconfirmed pass (tentative track matched)",
            '6_revive': "Stage 6a - Appearance REVIVAL  (force_id=best_id assigned)",
            '6_new':    "Stage 6b - New ID minted       (failed/skipped revival)",
        }
        for key, label in stage_labels.items():
            count = self.ids_by_stage.get(key, 0)
            pct = 100*count/total if total>0 else 0
            bar = '|' * int(pct/2)
            print(f"  {label}")
            print(f"    Count={count:4d} ({pct:5.1f}%)  {bar}")

            # Sub-stats for this stage
            stage_events = [e for e in self.ids_log if e['stage']==key]
            if stage_events:
                if key == 3:
                    # Show re-activation from lost stats
                    print(f"    (These are lost tracks re-matched via IoU - track_id unchanged)")
                elif key == '6_new':
                    # Break down by reason
                    reasons = defaultdict(int)
                    for e in stage_events: reasons[e['reason']] += 1
                    for r,c in sorted(reasons.items(), key=lambda x:-x[1]):
                        print(f"      {r}: {c}")
                    # Show score distribution
                    scores = [e['det_score'] for e in stage_events if e['det_score'] is not None]
                    if scores:
                        print(f"    Det score: min={min(scores):.3f} mean={np.mean(scores):.3f} max={max(scores):.3f}")
                elif key == '6_revive':
                    sims = [e['cosine_sim'] for e in stage_events if e['cosine_sim'] is not None]
                    thetas = [e['cosine_theta'] for e in stage_events if e['cosine_theta'] is not None]
                    margins = [e['cosine_margin'] for e in stage_events if e['cosine_margin'] is not None]
                    if sims:
                        print(f"    Cosine sim:   min={min(sims):.3f} mean={np.mean(sims):.3f} max={max(sims):.3f}")
                    if thetas:
                        print(f"    Theta:        min={min(thetas):.3f} mean={np.mean(thetas):.3f} max={max(thetas):.3f}")
                    if margins:
                        print(f"    Margin:       min={min(margins):.3f} mean={np.mean(margins):.3f} max={max(margins):.3f}")
            print()

        # Stage 6 gate failure breakdown
        gate_fails = [e for e in self.ids_log if 'gate_failed' in str(e.get('reason',''))]
        if gate_fails:
            sims_gf = [e['cosine_sim'] for e in gate_fails if e['cosine_sim'] is not None]
            thetas_gf = [e['cosine_theta'] for e in gate_fails if e['cosine_theta'] is not None]
            margins_gf = [e['cosine_margin'] for e in gate_fails if e['cosine_margin'] is not None]
            print(f"  Stage 6b gate failures ({len(gate_fails)} events):")
            if sims_gf:   print(f"    Best cosine sim:  min={min(sims_gf):.3f} mean={np.mean(sims_gf):.3f} max={max(sims_gf):.3f}")
            if thetas_gf: print(f"    Threshold theta:  min={min(thetas_gf):.3f} mean={np.mean(thetas_gf):.3f} max={max(thetas_gf):.3f}")
            if margins_gf:print(f"    Margin (sim-2nd): min={min(margins_gf):.3f} mean={np.mean(margins_gf):.3f} max={max(margins_gf):.3f}")
            print()

        print("=" * 70)
        print()
        print("First 25 IDS events (detailed):")
        print(f"  {'frame':>5}  {'stage':<10}  {'old_id':>6}  {'new_id':>6}  "
              f"{'det_score':>9}  {'iou_overlap':>11}  {'cosine_sim':>10}  "
              f"{'theta':>6}  {'margin':>7}  {'n_enrol':>7}  reason")
        print("  " + "-"*120)
        for e in self.ids_log[:25]:
            print(f"  {e['frame']:>5}  {str(e['stage']):<10}  {str(e['old_id']):>6}  {str(e['new_id']):>6}  "
                  f"{str(e['det_score']):>9}  {str(e['iou_overlap']):>11}  {str(e['cosine_sim']):>10}  "
                  f"{str(e['cosine_theta']):>6}  {str(e['cosine_margin']):>7}  {str(e['n_enrolments']):>7}  "
                  f"{e['reason']}")
        print()
        self.print_dlow_summary()
        self.print_gmc_adaptive_log()

    def print_fusion_summary(self):
        if not self.use_stage1_fusion: return
        s=self.fusion_stats
        print(f"\n  STAGE-1 FUSION SUMMARY (min IoU/appearance):")
        print(f"    frames with fusion active            : {s['frames_with_fusion']}")
        print(f"    mature-template track rows seen      : {s['mature_template_rows_seen']}")
        print(f"    cost cells overridden by appearance  : {s['cells_overridden']}")
        print(f"    Stage-1 matches decided by appearance: {s['matches_decided_by_appearance']}")
        print(f"    active fusion floor (final)          : {self._active_fusion_floor}")

    def print_gmc_adaptive_log(self):
        print()
        print('=' * 60)
        print('GMC + ADAPTIVE GATES SUMMARY')
        print('=' * 60)
        print(f'  GMC enabled        : {self.use_gmc}')
        print(f'  Adaptive gates     : {self.adaptive_gates}')
        if self._scene_log:
            for entry in self._scene_log:
                label = entry['scene_label'].upper()
                print(f'  Scene at frame {entry["frame"]:4d}: '
                      f'mean_pairwise_sim={entry["mean_pairwise_sim"]:.3f} '
                      f'[{label}] -> '
                      f'margin={entry["active_margin"]:.3f}  '
                      f'theta_cap={entry["active_theta_cap"]:.3f}')
        else:
            print('  Scene not yet estimated (warmup not reached)')
        gmc_used = self._gmc_matrix is not None
        print(f'  GMC last matrix    : {"applied" if gmc_used else "None (no prev frame)"}')
        print('=' * 60)
        print()

    def update(self, dets_xyxy, scores, image=None):
        self.frame_id += 1
        activated_stracks=[]; refind_stracks=[]; lost_stracks=[]; removed_stracks=[]

        # Stage 1: split
        dets_xyxy=np.asarray(dets_xyxy,dtype=float).reshape(-1,4)
        scores=np.asarray(scores,dtype=float).reshape(-1)
        inds_high=scores>self.track_thresh
        inds_low=(scores>0.1)&(scores<=self.track_thresh)
        dets_high=dets_xyxy[inds_high]; scores_high=scores[inds_high]
        dets_low=dets_xyxy[inds_low];   scores_low=scores[inds_low]
        detections=[STrack(STrack.tlbr_to_tlwh(b),s)
                    for b,s in zip(dets_high,scores_high)] if len(dets_high) else []
        detections_low=[STrack(STrack.tlbr_to_tlwh(b),s)
                        for b,s in zip(dets_low,scores_low)] if len(dets_low) else []

        # Stage 2: Kalman predict
        unconfirmed=[]; tracked_stracks=[]
        for t in self.tracked_stracks:
            if not t.is_activated: unconfirmed.append(t)
            else: tracked_stracks.append(t)
        strack_pool=joint_stracks(tracked_stracks,self.lost_stracks)
        STrack.multi_predict(strack_pool)

        # ---- GMC: camera motion compensation ----
        # Estimate affine transform between prev and curr frame.
        # Correct all Kalman-predicted centres so IoU matching is
        # camera-motion-agnostic (BoT-SORT style).
        if self.use_gmc and image is not None:
            curr_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            self._gmc_matrix = estimate_affine_gmc(self._prev_gray, curr_gray)
            if self._gmc_matrix is not None:
                apply_gmc_to_stracks(strack_pool, self._gmc_matrix)
            self._prev_gray = curr_gray

        # ---- Scene-adaptive gates ----
        # At warmup frame, compute pairwise template similarity to estimate
        # scene difficulty and adapt margin + theta_cap accordingly.
        if (self.adaptive_gates
                and self._scene_difficulty is None
                and self.frame_id == self.adaptive_warmup_frames):
            diff = self.registry.compute_scene_difficulty()
            if diff is not None:
                self._scene_difficulty = diff
                # Interpolate linearly between hard and easy endpoints
                # diff < 0.45 -> easy;  diff > 0.60 -> hard;  else linear blend
                t = max(0.0, min(1.0, (diff - 0.45) / 0.15))
                self._active_margin = (
                    (1-t) * self.adaptive_margin_easy
                    + t   * self.adaptive_margin_hard)
                self._active_theta_cap = (
                    (1-t) * self.adaptive_cap_easy
                    + t   * self.adaptive_cap_hard)
                if self.use_stage1_fusion and self.fusion_adaptive:
                    self._active_fusion_floor = (
                        (1-t) * self.fusion_floor_easy
                        + t   * self.fusion_floor_hard)
                self._scene_log.append({
                    'frame': self.frame_id,
                    'mean_pairwise_sim': round(diff, 4),
                    'scene_label': 'hard' if diff > 0.60 else ('easy' if diff < 0.45 else 'medium'),
                    'active_margin': round(self._active_margin, 4),
                    'active_theta_cap': round(self._active_theta_cap, 4),
                    'active_fusion_floor': round(self._active_fusion_floor, 4),
                })
                print(f'   [adaptive gates] frame={self.frame_id} '
                      f'mean_sim={diff:.3f} -> '
                      f'margin={self._active_margin:.3f} '
                      f'theta_cap={self._active_theta_cap:.3f}')

        # Stage 3: FIRST ASSOCIATION -- compute and SAVE IoU matrices
        raw_iou = iou_distance(strack_pool, detections) if detections else np.empty((0,0))
        fused = fuse_score(raw_iou.copy(), detections) if (not self.mot20 and detections) else raw_iou.copy()
        # ---- Test-8: Stage-1 fusion -- min(IoU, appearance) cost --------------
        # Appearance enters the FIRST association (where 92.8% of ID events
        # occur, at IoU>0.5 vs a neighbor). For every (track, det) pair that is
        # (a) spatially plausible (IoU cost <= fusion_proximity_thresh) and
        # (b) above the scene-adaptive appearance floor against the track's
        # frozen-median template, cost becomes min(fused_iou, (1-cos)/2).
        # Tracks without a mature template keep pure IoU (emb row stays 1.0).
        _fusion_emb=None; _fusion_pre=None
        self._stage1_det_feats=None
        if (self.use_stage1_fusion and image is not None
                and len(detections)>0 and len(strack_pool)>0):
            _crops=[]; _keep=[]
            for _j,_det in enumerate(detections):
                _c=crop_from_image(image,_det.tlbr)
                if _c is not None: _crops.append(_c); _keep.append(_j)
            if _crops:
                _feats=self.reid.extract(_crops)        # (m,512) L2-normalised
                self._stage1_det_feats=np.full((len(detections),512),np.nan,
                                               dtype=np.float32)
                for _j,_f in zip(_keep,_feats): self._stage1_det_feats[_j]=_f
                _floor=(self._active_fusion_floor
                        if self._active_fusion_floor is not None
                        else self.fusion_sim_floor)
                _emb=np.ones_like(fused)
                _n_mature=0
                with np.errstate(invalid='ignore'):
                    for _i,_trk in enumerate(strack_pool):
                        _e=self.registry.entries.get(_trk.track_id)
                        if (_e is None or
                                _e['n']<self.registry.MIN_TEMPLATE_N_FOR_QUERY):
                            continue
                        _n_mature+=1
                        _sims=self._stage1_det_feats@_e['mean']
                        _cost=(1.0-_sims)/2.0
                        _cost=np.where(np.isnan(_cost),1.0,_cost)
                        _cost=np.where(_sims<_floor,1.0,_cost)
                        _emb[_i]=_cost
                _emb[raw_iou>self.fusion_proximity_thresh]=1.0
                _fusion_pre=fused.copy()
                fused=np.minimum(fused,_emb)
                _fusion_emb=_emb
                self.fusion_stats['frames_with_fusion']+=1
                self.fusion_stats['mature_template_rows_seen']+=_n_mature
                self.fusion_stats['cells_overridden']+=int(
                    (fused<_fusion_pre-1e-9).sum())

        matches,u_track,u_det = linear_assignment(fused, thresh=self.match_thresh) if detections else                                  (np.empty((0,2),dtype=int), tuple(range(len(strack_pool))), tuple())

        if _fusion_emb is not None and len(matches)>0:
            for _it,_idt in matches:
                if _fusion_emb[_it,_idt] < _fusion_pre[_it,_idt]-1e-9:
                    self.fusion_stats['matches_decided_by_appearance']+=1

        ids_this_frame = []  # track which frames had IDS events for matrix saving

        for itracked,idet in matches:
            track=strack_pool[itracked]; det=detections[idet]
            if track.state==TrackState.Tracked:
                track.update(det,self.frame_id)
                activated_stracks.append(track)
            else:
                # Lost track re-activated via IoU -- log with full diagnostics
                old_id=track.track_id
                iou_c = float(raw_iou[itracked, idet]) if raw_iou.size > 0 else None
                fused_c = float(fused[itracked, idet]) if fused.size > 0 else None
                track.re_activate(det,self.frame_id,new_id=False)
                self._log_event(
                    self.frame_id, 3, old_id, track.track_id,
                    're_activate_from_lost',
                    det_score=float(det.score),
                    det_box=det.tlbr.tolist(),
                    track_box=track.tlbr.tolist(),
                    iou_cost=iou_c,
                    fused_cost=fused_c,
                    n_registry=len([e for e in self.registry.entries.values()
                                    if e['n']>=self.registry.MIN_TEMPLATE_N_FOR_QUERY]),
                    n_lost=len(self.lost_stracks),
                    n_enrolments=self.registry.get_n_enrolments(old_id),
                )
                ids_this_frame.append(self.frame_id)
                refind_stracks.append(track)

        # Save IoU matrix for frames where Stage 3 IDS occurred
        if ids_this_frame and detections and strack_pool:
            self._save_iou_matrix(self.frame_id, strack_pool, detections, raw_iou, fused)

        # Stage 3.5: template enrolment
        if image is not None and len(matches)>0:
            if self._stage1_det_feats is not None:
                # Test-8: reuse Stage-1 fusion features -- no second OSNet pass
                for itracked,idet in matches:
                    feat=self._stage1_det_feats[idet]
                    if np.isnan(feat[0]): continue
                    self.registry.update(strack_pool[itracked].track_id,
                                         feat,self.frame_id)
            else:
                enrol_crops=[]; enrol_track_ids=[]
                for itracked,idet in matches:
                    track=strack_pool[itracked]; det=detections[idet]
                    crop=crop_from_image(image,det.tlbr)
                    if crop is None: continue
                    enrol_crops.append(crop); enrol_track_ids.append(track.track_id)
                if enrol_crops:
                    feats=self.reid.extract(enrol_crops)
                    for tid,feat in zip(enrol_track_ids,feats):
                        self.registry.update(tid,feat,self.frame_id)

        # Stage 4: unmatched -> lost
        for it in u_track:
            track=strack_pool[it]
            if track.state!=TrackState.Lost:
                track.mark_lost(); lost_stracks.append(track)

        # ---- STAGE 4.5: D_LOW RESCUE (Rec 3, ByteTrack second association) ----
        # Try to keep just-lost tracks alive using low-confidence (D_low)
        # detections. We associate D_low dets to the tracks marked lost THIS
        # frame by IoU; any match with IoU >= dlow_iou_thresh re-activates the
        # track WITHOUT a new id (identity preserved -> no IDS). Rescued
        # detections are removed from the discard analysis below.
        rescued_dlow_idx = set()
        rescued_lost_ids = set()
        if (self.use_dlow_rescue and not self.use_occlusion_reid
                and len(detections_low) > 0 and len(lost_stracks) > 0):
            dists_dlow = iou_distance(lost_stracks, detections_low)  # cost = 1 - IoU
            m_dlow, _u_lost, _u_dlow = linear_assignment(
                dists_dlow, thresh=1.0 - self.dlow_iou_thresh)
            for it, idd in m_dlow:
                track = lost_stracks[it]
                dlow  = detections_low[idd]
                iou_overlap = 1.0 - float(dists_dlow[it, idd])
                old_id = track.track_id
                track.re_activate(dlow, self.frame_id, new_id=False)
                refind_stracks.append(track)
                rescued_dlow_idx.add(int(idd))
                rescued_lost_ids.add(old_id)
                self.dlow_summary['dlow_rescued'] += 1
                self._log_event(
                    self.frame_id, '4_dlow_rescue', old_id, track.track_id,
                    f'dlow_rescue iou={iou_overlap:.3f} score={float(dlow.score):.3f}',
                    det_score=float(dlow.score),
                    det_box=dlow.tlbr.tolist(),
                    track_box=track.tlbr.tolist(),
                    iou_cost=float(dists_dlow[it, idd]),
                    n_lost=len(self.lost_stracks),
                    n_enrolments=self.registry.get_n_enrolments(old_id),
                )
            # Drop rescued tracks from the lost list for this frame.
            if rescued_lost_ids:
                lost_stracks = [t for t in lost_stracks
                                if t.track_id not in rescued_lost_ids]

        # ---- D_LOW DISCARD ANALYSIS ----
        # At this point lost_stracks contains the tracks just marked lost
        # this frame. We now check every D_low detection to see:
        #   (a) its score and box
        #   (b) whether it overlaps any just-lost track (IoU > 0.3)
        #       -> those are people we discarded that could have kept a
        #          track alive (the ByteTrack second-pass would catch them)
        if len(detections_low) > 0:
            self.dlow_summary['frames_with_dlow'] += 1
            self.dlow_summary['total_dlow_dets'] += len(detections_low)

            # Compute IoU between D_low dets and just-lost tracks
            newly_lost = lost_stracks  # tracks marked lost this very frame
            if newly_lost:
                iou_dlow_lost = iou_distance(newly_lost, detections_low)  # (n_lost x n_dlow)
                iou_overlap_dlow = 1 - iou_dlow_lost  # convert cost -> overlap
            else:
                iou_overlap_dlow = np.zeros((0, len(detections_low)))

            for j, dlow in enumerate(detections_low):
                was_rescued = j in rescued_dlow_idx
                # Find best overlapping lost track for this D_low detection
                # (newly_lost no longer contains rescued tracks).
                if iou_overlap_dlow.shape[0] > 0:
                    overlaps = iou_overlap_dlow[:, j]
                    best_overlap_idx = int(np.argmax(overlaps))
                    best_overlap = float(overlaps[best_overlap_idx])
                    best_lost_id  = newly_lost[best_overlap_idx].track_id if best_overlap > 0.01 else None
                    best_lost_box = newly_lost[best_overlap_idx].tlbr.tolist() if best_overlap > 0.01 else None
                else:
                    best_overlap = 0.0; best_lost_id = None; best_lost_box = None

                # Rescued dets were already counted under 'dlow_rescued' in the
                # rescue pass; only the STILL-discarded dets feed the overlap/bg
                # bookkeeping here.
                could_rescue = (not was_rescued) and best_overlap >= 0.3
                if not was_rescued:
                    if best_overlap >= 0.3:
                        self.dlow_summary['dlow_overlapping_lost_track'] += 1
                    else:
                        self.dlow_summary['dlow_no_overlap'] += 1

                self.dlow_log.append({
                    'frame':           self.frame_id,
                    'det_score':       round(float(dlow.score), 4),
                    'det_box':         [round(v,1) for v in dlow.tlbr.tolist()],
                    'best_lost_track_id':  best_lost_id,
                    'best_lost_track_box': best_lost_box,
                    'iou_overlap_with_lost': round(best_overlap, 4),
                    'could_rescue_lost_track': could_rescue,
                    'rescued':         was_rescued,
                    'n_dlow_this_frame': len(detections_low),
                    'n_newly_lost_this_frame': len(newly_lost),
                })

        # Stage 5: unconfirmed pass
        detections_remain=[detections[i] for i in u_det]
        dists5=iou_distance(unconfirmed,detections_remain)
        if not self.mot20: dists5=fuse_score(dists5,detections_remain)
        matches5,u_unconfirmed,u_det2=linear_assignment(dists5,thresh=0.7)
        for itracked,idet in matches5:
            old_id=unconfirmed[itracked].track_id
            det=detections_remain[idet]
            iou_c5=float(iou_distance([unconfirmed[itracked]],[det])[0,0])
            unconfirmed[itracked].update(det,self.frame_id)
            self._log_event(
                self.frame_id, 5, old_id, unconfirmed[itracked].track_id,
                'unconfirmed_update',
                det_score=float(det.score),
                det_box=det.tlbr.tolist(),
                track_box=unconfirmed[itracked].tlbr.tolist(),
                iou_cost=iou_c5,
                n_lost=len(self.lost_stracks),
            )
            activated_stracks.append(unconfirmed[itracked])
        for it in u_unconfirmed:
            unconfirmed[it].mark_removed(); removed_stracks.append(unconfirmed[it])

        # Stage 6: APPEARANCE RECOVERY with full diagnostics
        still_unmatched=[detections_remain[i] for i in u_det2]
        n_mature = len([e for e in self.registry.entries.values()
                        if e['n']>=self.registry.MIN_TEMPLATE_N_FOR_QUERY])
        n_lost_now = len(self.lost_stracks)

        if image is not None and len(still_unmatched)>0 and len(self.registry.entries)>0:
            crops=[crop_from_image(image,d.tlbr) for d in still_unmatched]
            valid_idx=[i for i,c in enumerate(crops) if c is not None]
            if valid_idx:
                feats=self.reid.extract([crops[i] for i in valid_idx])
                for local_i,det_i in enumerate(valid_idx):
                    d=still_unmatched[det_i]
                    if d.score<self.det_thresh: continue
                    feat=feats[local_i]
                    feat_norm=float(np.linalg.norm(feat))

                    best_id,best_sim,second_sim,sim_map=self.registry.query(feat)

                    if best_id is None:
                        d.activate(self.kalman_filter,self.frame_id)
                        if n_lost_now>0:
                            self._log_event(
                                self.frame_id, '6_new', -1, d.track_id,
                                'registry_empty_or_immature',
                                det_score=float(d.score),
                                det_box=d.tlbr.tolist(),
                                feat_norm=feat_norm,
                                n_registry=n_mature, n_lost=n_lost_now,
                            )
                        activated_stracks.append(d); continue

                    theta=self.registry.acceptance_threshold(best_id,global_floor=self.reid_global_floor,theta_cap=self._active_theta_cap)
                    margin=best_sim-second_sim
                    n_enrol=self.registry.get_n_enrolments(best_id)

                    # Compute IoU between this detection and the best matching track (if in strack_pool)
                    best_track_box=None
                    best_track_in_pool=[t for t in self.lost_stracks if t.track_id==best_id]
                    if best_track_in_pool:
                        best_track_box=best_track_in_pool[0].tlbr.tolist()
                        iou_c6=float(iou_distance([best_track_in_pool[0]],[d])[0,0])
                    else:
                        iou_c6=None

                    # Occlusion-aware motion gate: a reappearing detection must
                    # land near the lost track's motion-predicted position, with
                    # tolerance growing the longer the track has been occluded.
                    motion_ok=True; motion_dist=None; motion_gate=None
                    if self.use_occlusion_reid and best_track_in_pool:
                        _lt=best_track_in_pool[0]
                        _pb=_lt.tlwh
                        _pcx=_pb[0]+_pb[2]/2.0; _pcy=_pb[1]+_pb[3]/2.0
                        _db=d.tlwh
                        _dcx=_db[0]+_db[2]/2.0; _dcy=_db[1]+_db[3]/2.0
                        motion_dist=float(np.hypot(_pcx-_dcx,_pcy-_dcy))
                        _diag=float(np.hypot(_pb[2],_pb[3]))+1e-6
                        _flost=max(0,self.frame_id-_lt.end_frame())
                        motion_gate=_diag*(self.occ_motion_gate_base
                                           +self.occ_motion_gate_growth*_flost)
                        motion_ok=motion_dist<=motion_gate

                    if best_sim>=theta and margin>=self._active_margin and motion_ok:
                        # REVIVE
                        d.activate(self.kalman_filter,self.frame_id,force_id=best_id)
                        self._log_event(
                            self.frame_id, '6_revive', best_id, d.track_id,
                            f'revived',
                            det_score=float(d.score),
                            det_box=d.tlbr.tolist(),
                            track_box=best_track_box,
                            iou_cost=iou_c6,
                            cosine_sim=best_sim,
                            cosine_theta=theta,
                            cosine_margin=margin,
                            feat_norm=feat_norm,
                            feat_mean_sim=best_sim,
                            n_registry=n_mature, n_lost=n_lost_now,
                            n_enrolments=n_enrol,
                            all_sims=sim_map,
                        )
                        self.registry.update(best_id,feat,self.frame_id)
                    else:
                        # GATE FAILED -- mint new ID
                        d.activate(self.kalman_filter,self.frame_id)
                        if n_lost_now>0:
                            if best_sim<theta:
                                reason=f'gate_failed sim={best_sim:.3f}<theta={theta:.3f}'
                            elif margin<self._active_margin:
                                reason=f'margin_failed margin={margin:.3f}<{self._active_margin:.2f}'
                            else:
                                reason=f'motion_failed dist={motion_dist:.0f}>gate={motion_gate:.0f}'
                            self._log_event(
                                self.frame_id, '6_new', best_id, d.track_id,
                                reason,
                                det_score=float(d.score),
                                det_box=d.tlbr.tolist(),
                                track_box=best_track_box,
                                iou_cost=iou_c6,
                                cosine_sim=best_sim,
                                cosine_theta=theta,
                                cosine_margin=margin,
                                feat_norm=feat_norm,
                                n_registry=n_mature, n_lost=n_lost_now,
                                n_enrolments=n_enrol,
                                all_sims=sim_map,
                            )
                    activated_stracks.append(d)
                for det_i in range(len(still_unmatched)):
                    if det_i in valid_idx: continue
                    d=still_unmatched[det_i]
                    if d.score<self.det_thresh: continue
                    d.activate(self.kalman_filter,self.frame_id)
                    activated_stracks.append(d)
            else:
                for d in still_unmatched:
                    if d.score<self.det_thresh: continue
                    d.activate(self.kalman_filter,self.frame_id)
                    activated_stracks.append(d)
        else:
            for d in still_unmatched:
                if d.score<self.det_thresh: continue
                d.activate(self.kalman_filter,self.frame_id)
                activated_stracks.append(d)

        # Stage 7: age out
        for track in self.lost_stracks:
            if self.frame_id-track.end_frame()>self.max_time_lost:
                track.mark_removed(); removed_stracks.append(track)

        # Stage 8: update lists
        self.tracked_stracks=[t for t in self.tracked_stracks if t.state==TrackState.Tracked]
        self.tracked_stracks=joint_stracks(self.tracked_stracks,activated_stracks)
        self.tracked_stracks=joint_stracks(self.tracked_stracks,refind_stracks)
        self.lost_stracks=sub_stracks(self.lost_stracks,self.tracked_stracks)
        self.lost_stracks.extend(lost_stracks)
        self.lost_stracks=sub_stracks(self.lost_stracks,self.removed_stracks)
        self.removed_stracks.extend(removed_stracks)
        self.tracked_stracks,self.lost_stracks=remove_duplicate_stracks(
            self.tracked_stracks,self.lost_stracks)

        return [t for t in self.tracked_stracks if t.is_activated]


print("TR-YOLO11x v1 + Full Diagnostic Logger loaded.")
print("  After tracking call:")
print("    tracker.print_ids_summary()           -- console breakdown")
print("    tracker.save_ids_csv(path)            -- all events to CSV")
print("    tracker.save_iou_matrices(output_dir) -- per-frame IoU CSVs")
print("  Access raw data:")
print("    tracker.ids_log                       -- list of dicts")
print("    tracker.ids_by_stage                  -- stage -> count")
print("    tracker.iou_matrix_frames             -- frame -> matrix dict")


In [ ]:
# ============================================================================
# CELL 4: Helper functions — load one sequence, track it, score MOTA + HOTA
# ============================================================================
# These wrap the logic from the original single-sequence cells (3, 5, 6, 8, 9)
# into functions the driver loop calls once per sequence. The tracking and
# evaluation logic is unchanged — only refactored to take a sequence name.

import os, glob, time, configparser, shutil, tempfile, warnings, subprocess, sys
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
import torch as _torch

# --- NumPy 2.0 compat shim for motmetrics 1.4.0 (np.asfarray removed) -------
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)
import motmetrics as mm

# --- NumPy alias patch for TrackEval, then import it -------------------------
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for _name, _typ in (('float', float), ('int', int), ('bool', bool),
                        ('object', object), ('str', str), ('long', int)):
        if not hasattr(np, _name):
            setattr(np, _name, _typ)
try:
    import trackeval
except ImportError:
    print('Installing TrackEval (one-time) ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'git+https://github.com/JonathonLuiten/TrackEval.git'])
    import trackeval


# ---- Load YOLO11x once (private detection) ---------------------------------
_YOLO      = None
_YOLO_FP16 = False
_YOLO_DEV  = 'cpu'
if USE_YOLO:
    from yolox.models import YOLOX as _YX, YOLOPAFPN as _YXFPN, YOLOXHead as _YXHead
    from yolox.utils import postprocess as _yolox_postprocess
    print(f'Loading YOLOX-X (ByteTrack) for private detection: {YOLOX_WEIGHTS}')
    _in_ch = [256, 512, 1024]
    _YOLO  = _YX(_YXFPN(YOLOX_DEPTH, YOLOX_WIDTH, in_channels=_in_ch),
                 _YXHead(YOLOX_NUM_CLS, YOLOX_WIDTH, in_channels=_in_ch))
    _ckpt  = _torch.load(YOLOX_WEIGHTS, map_location='cpu')
    _YOLO.load_state_dict(_ckpt.get('model', _ckpt), strict=True)
    _YOLO_DEV  = 'cuda' if _torch.cuda.is_available() else 'cpu'
    _YOLO_FP16 = (_YOLO_DEV == 'cuda')
    _YOLO.to(_YOLO_DEV).eval()
    if _YOLO_FP16:
        _YOLO.half()
    print(f'   YOLOX ready (device={_YOLO_DEV}, fp16={_YOLO_FP16}, '
          f'tsize={YOLOX_TSIZE}, batch={YOLO_BATCH})')

def _yolox_preproc(img_bgr):
    """ByteTrack preprocessing — MUST match the trained weights:
    letterbox to YOLOX_TSIZE (H,W) padding 114, BGR->RGB, /255, subtract
    ImageNet mean, divide by std, then HWC->CHW. (Verified against ByteTrack's
    yolox/data/data_augment.py preproc.)"""
    th, tw = YOLOX_TSIZE
    padded = np.ones((th, tw, 3), dtype=np.float32) * 114.0
    r = min(th / img_bgr.shape[0], tw / img_bgr.shape[1])
    rh, rw = int(img_bgr.shape[0] * r), int(img_bgr.shape[1] * r)
    padded[:rh, :rw] = cv2.resize(
        img_bgr, (rw, rh), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    padded = padded[:, :, ::-1]                       # BGR -> RGB
    padded /= 255.0
    padded -= np.asarray(YOLOX_RGB_MEAN, np.float32)
    padded /= np.asarray(YOLOX_RGB_STD,  np.float32)
    padded = padded.transpose(2, 0, 1)                # HWC -> CHW
    return np.ascontiguousarray(padded, dtype=np.float32), r

def yolo_detect_batch(frames_bgr):
    """Run YOLOX-X on a list of BGR frames in one GPU call.
    Returns list of Nx5 arrays [x,y,w,h,conf] (tlwh), one per frame — the same
    output contract as the previous YOLO11x detector, so the tracker is unchanged."""
    pre, ratios = [], []
    for f in frames_bgr:
        p, r = _yolox_preproc(f); pre.append(p); ratios.append(r)
    batch = _torch.from_numpy(np.stack(pre, 0)).to(_YOLO_DEV)
    if _YOLO_FP16:
        batch = batch.half()
    with _torch.no_grad():
        preds = _YOLO(batch)
        preds = _yolox_postprocess(preds, YOLOX_NUM_CLS,
                                   YOLOX_PP_CONF, YOLOX_NMS, class_agnostic=True)
    out = []
    for det, r in zip(preds, ratios):
        if det is None or len(det) == 0:
            out.append(np.empty((0, 5), np.float32)); continue
        det   = det.cpu().numpy()
        boxes = det[:, :4] / r                        # map back to original scale
        score = det[:, 4] * det[:, 5]                 # obj_conf * class_conf
        keep  = score >= YOLOX_CONF
        boxes, score = boxes[keep], score[keep]
        arr = np.zeros((len(boxes), 5), np.float32)
        arr[:, 0] = boxes[:, 0]; arr[:, 1] = boxes[:, 1]
        arr[:, 2] = boxes[:, 2] - boxes[:, 0]         # w
        arr[:, 3] = boxes[:, 3] - boxes[:, 1]         # h
        arr[:, 4] = score
        out.append(arr)
    return out

def _load_frames(paths):
    """OPT: load frames on background thread while GPU runs YOLO."""
    return [cv2.imread(p) for p in paths]


# ---------------------------------------------------------------------------
def load_sequence(seq_name):
    """Resolve paths + read seqinfo.ini for one sequence.
    Returns a dict of everything the tracker/eval need, or raises if a
    required file is missing."""
    seq_dir     = os.path.join(MOT17_TRAIN_DIR, seq_name)
    frames_dir  = os.path.join(seq_dir, 'img1')
    gt_file     = os.path.join(seq_dir, 'gt',  'gt.txt')
    seqinfo_ini = os.path.join(seq_dir, 'seqinfo.ini')

    # Private detection (YOLO11x): det.txt is never used.
    required = [(frames_dir, 'frames'), (seqinfo_ini, 'seqinfo')]  # TEST: no gt/
    for p, label in required:
        if not os.path.exists(p):
            raise FileNotFoundError(f'{seq_name}: missing {label}: {p}')

    cfg = configparser.ConfigParser()
    cfg.read(seqinfo_ini)
    seq_length = int(cfg['Sequence']['seqLength'])
    fps        = int(cfg['Sequence']['frameRate'])
    im_width   = int(cfg['Sequence']['imWidth'])
    im_height  = int(cfg['Sequence']['imHeight'])
    im_ext     = cfg['Sequence']['imExt']

    return {
        'seq_name': seq_name, 'seq_dir': seq_dir,
        'frames_dir': frames_dir, 'gt_file': gt_file,
        'seq_length': seq_length, 'fps': fps,
        'im_width': im_width, 'im_height': im_height, 'im_ext': im_ext,
    }


# ---------------------------------------------------------------------------
def run_tracking(seq, output_txt, render_mp4=False, output_mp4=None):
    """Run the NO_SAM BYTETracker over one sequence, write MOT-format output,
    flush the registry, optionally render an MP4. Returns (results,
    per_frame_active, tracker)."""
    frames_dir = seq['frames_dir']
    im_ext     = seq['im_ext']
    seq_length = seq['seq_length']
    fps        = seq['fps']

    def _keep_output(tlwh):
        x, y, w, h = tlwh
        if w * h < MIN_BOX_AREA:
            return False
        if w / max(h, 1e-6) > ASPECT_RATIO_THRESH:
            return False
        return True

    frame_files = sorted(glob.glob(os.path.join(frames_dir, f'*{im_ext}')))
    assert len(frame_files) == seq_length, \
        f'{seq["seq_name"]}: expected {seq_length} frames, found {len(frame_files)}'

    tracker = BYTETracker(
        track_thresh = TRACK_THRESH,
        match_thresh = MATCH_THRESH,
        track_buffer = TRACK_BUFFER,
        mot20        = MOT20_FLAG,
        frame_rate   = fps,
        device       = REID_DEVICE,
        reid_global_floor = REID_GLOBAL_FLOOR,
        reid_margin       = REID_MARGIN,
        reid_theta_cap    = REID_THETA_CAP,
        use_dlow_rescue   = USE_DLOW_RESCUE,
        dlow_iou_thresh   = DLOW_IOU_THRESH,
        # Test-6: new params
        use_gmc                = USE_GMC,
        adaptive_gates         = ADAPTIVE_GATES,
        adaptive_warmup_frames = ADAPTIVE_WARMUP_FRAMES,
        adaptive_margin_easy   = ADAPTIVE_MARGIN_EASY,
        adaptive_margin_hard   = ADAPTIVE_MARGIN_HARD,
        adaptive_cap_easy      = ADAPTIVE_CAP_EASY,
        adaptive_cap_hard      = ADAPTIVE_CAP_HARD,
        # Occlusion-aware re-ID
        use_occlusion_reid     = USE_OCCLUSION_REID,
        occ_motion_gate_base   = OCC_MOTION_GATE_BASE,
        occ_motion_gate_growth = OCC_MOTION_GATE_GROWTH,
        # Test-8: Stage-1 fusion
        use_stage1_fusion       = USE_STAGE1_FUSION,
        fusion_proximity_thresh = FUSION_PROXIMITY_THRESH,
        fusion_sim_floor        = FUSION_SIM_FLOOR,
        fusion_adaptive         = FUSION_ADAPTIVE,
        fusion_floor_easy       = FUSION_FLOOR_EASY,
        fusion_floor_hard       = FUSION_FLOOR_HARD,
        output_dir             = OUTPUT_DIR,
    )

    if _YOLO is None:
        raise RuntimeError(
            f'{seq["seq_name"]}: YOLO11x is not loaded. This pipeline is private-'
            'detection only (det.txt support removed). Set USE_YOLO=True and '
            're-run the init cell so _YOLO is available.')

    # OPT: accumulate in memory, single Drive write at end
    results          = []
    per_frame_active = [[] for _ in range(seq_length)]
    batches          = [frame_files[i:i+YOLO_BATCH]
                        for i in range(0, seq_length, YOLO_BATCH)]
    t0           = time.time()
    executor     = ThreadPoolExecutor(max_workers=2)
    prefetch_fut = executor.submit(_load_frames, batches[0])

    for batch_idx, batch_paths in enumerate(
            tqdm(batches, desc=f'Track {seq["seq_name"]}',
                 unit='batch', leave=True)):

        frames_bgr = prefetch_fut.result()
        if batch_idx + 1 < len(batches):
            prefetch_fut = executor.submit(_load_frames, batches[batch_idx + 1])

        valid = [(i, f) for i, f in enumerate(frames_bgr) if f is not None]
        if not valid:
            continue

        batch_dets = yolo_detect_batch([f for _, f in valid])

        for local_i, (frame_local_idx, frame_img) in enumerate(valid):
            frame_id = batch_idx * YOLO_BATCH + frame_local_idx + 1
            if frame_id > seq_length:
                break
            dets = batch_dets[local_i]
            if len(dets):
                tlwh   = dets[:, :4].astype(np.float32)
                tlbr   = tlwh.copy(); tlbr[:, 2:] += tlbr[:, :2]
                scores = dets[:, 4].astype(np.float32)
            else:
                tlbr   = np.empty((0, 4), np.float32)
                scores = np.empty((0,),   np.float32)

            online = tracker.update(tlbr, scores, image=frame_img)

            for t in online:
                tlwh_out = t.tlwh
                if not _keep_output(tlwh_out):
                    continue
                x, y, w, h = tlwh_out
                results.append((frame_id, t.track_id,
                                x, y, w, h, t.score, -1, -1, -1))
                per_frame_active[frame_id - 1].append(
                    (t.track_id, tlwh_out, t.score))

    executor.shutdown(wait=False)
    elapsed  = time.time() - t0
    n_tracks = len({r[1] for r in results})

    # Write MOT-format output.
    with open(output_txt, 'w') as f:
        for row in results:
            f.write(f'{int(row[0])},{int(row[1])},{row[2]:.2f},{row[3]:.2f},'
                    f'{row[4]:.2f},{row[5]:.2f},{row[6]:.4f},-1,-1,-1\n')

    # Flush appearance-template registry next to the output.
    registry_csv = os.path.splitext(output_txt)[0] + '_registry.csv'
    tracker.flush_registry(registry_csv)
    tracker.print_ids_summary()
    tracker.print_fusion_summary()
    ids_csv = os.path.join(OUTPUT_DIR, f'{seq_name}_ids_events.csv')
    tracker.save_ids_csv(ids_csv)
    tracker.save_iou_matrices(OUTPUT_DIR)
    dlow_csv = os.path.join(OUTPUT_DIR, f'{seq_name}_dlow_discards.csv')
    tracker.save_dlow_csv(dlow_csv)

    print(f'   tracked {seq_length}f in {elapsed:.1f}s '
          f'({seq_length/elapsed:.1f} FPS) | rows={len(results)} '
          f'| IDs={n_tracks} | registry={len(tracker.registry.entries)}')

    # Optional MP4.
    if render_mp4 and output_mp4 is not None:
        _render_mp4(seq, frame_files, per_frame_active, output_mp4)

    return results, per_frame_active, tracker


# ---------------------------------------------------------------------------
def _render_mp4(seq, frame_files, per_frame_active, output_mp4):
    def _color_for_id(tid):
        h = (tid * 37) % 180
        hsv = np.uint8([[[h, 220, 255]]])
        return tuple(int(c) for c in cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0, 0])
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_mp4, fourcc, seq['fps'],
                             (seq['im_width'], seq['im_height']))
    for i, path in enumerate(tqdm(frame_files, desc='   MP4', leave=False)):
        frame = cv2.imread(path)
        if frame is None:
            continue
        for tid, tlwh, score in per_frame_active[i]:
            x, y, w, h = tlwh
            x1, y1, x2, y2 = int(x), int(y), int(x + w), int(y + h)
            color = _color_for_id(tid)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label = f'ID {tid}  {score:.2f}'
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
            cv2.putText(frame, label, (x1 + 2, y1 - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
        writer.write(frame)
    writer.release()


# ---------------------------------------------------------------------------
def eval_mota(seq, output_txt, mota_csv):
    """Score one sequence with py-motmetrics. Writes the full summary row to
    mota_csv and returns a dict of the headline numbers."""
    gt_df_full = pd.read_csv(seq['gt_file'], header=None).iloc[:, :9].copy()
    gt_df_full.columns = ['frame','id','x','y','w','h','conf','class','vis']
    gt = gt_df_full[(gt_df_full['class'] == 1) & (gt_df_full['conf'] == 1)].copy()

    if (not os.path.exists(output_txt)) or os.path.getsize(output_txt) == 0:
        trk = pd.DataFrame(columns=['frame','id','x','y','w','h','score'])
    else:
        trk = pd.read_csv(output_txt, header=None).iloc[:, :7].copy()
        trk.columns = ['frame','id','x','y','w','h','score']

    acc = mm.MOTAccumulator(auto_id=True)
    all_frames = sorted(set(gt['frame'].unique()) | set(trk['frame'].unique()))
    for f in all_frames:
        gt_f, trk_f = gt[gt['frame'] == f], trk[trk['frame'] == f]
        dists = mm.distances.iou_matrix(
            gt_f[['x','y','w','h']].values,
            trk_f[['x','y','w','h']].values, max_iou=0.5)
        acc.update(gt_f['id'].astype(int).tolist(),
                   trk_f['id'].astype(int).tolist(), dists)

    mh = mm.metrics.create()
    summary = mh.compute(
        acc,
        metrics=['mota','idf1','idp','idr','recall','precision',
                 'num_unique_objects','mostly_tracked','mostly_lost',
                 'num_switches','num_fragmentations',
                 'num_false_positives','num_misses','num_objects'],
        name=seq['seq_name'])
    summary.to_csv(mota_csv)

    return {
        'MOTA':   float(summary['mota'].iloc[0]) * 100,
        'IDF1':   float(summary['idf1'].iloc[0]) * 100,
        'IDs':    int(summary['num_switches'].iloc[0]),
        'FP':     int(summary['num_false_positives'].iloc[0]),
        'FN':     int(summary['num_misses'].iloc[0]),
        'MT':     int(summary['mostly_tracked'].iloc[0]),
        'ML':     int(summary['mostly_lost'].iloc[0]),
        'Frag':   int(summary['num_fragmentations'].iloc[0]),
        'Recall': float(summary['recall'].iloc[0]) * 100,
        'Precision': float(summary['precision'].iloc[0]) * 100,
    }


# ---------------------------------------------------------------------------
def eval_hota(seq, output_txt, summary_csv, per_alpha_csv):
    """Score one sequence with TrackEval HOTA. Writes summary + per-alpha CSVs
    and returns a dict of mean HOTA metrics."""
    seq_name   = seq['seq_name']
    seq_length = seq['seq_length']
    BENCHMARK, SPLIT, TRACKER = 'MOT17', 'train', 'tracker'

    work     = tempfile.mkdtemp(prefix='trackeval_')
    gt_root  = os.path.join(work, 'gt', BENCHMARK)
    trk_root = os.path.join(work, 'trackers', BENCHMARK)

    gt_seq_dir = os.path.join(gt_root, f'{BENCHMARK}-{SPLIT}', seq_name)
    os.makedirs(os.path.join(gt_seq_dir, 'gt'), exist_ok=True)
    shutil.copy(seq['gt_file'], os.path.join(gt_seq_dir, 'gt', 'gt.txt'))
    with open(os.path.join(gt_seq_dir, 'seqinfo.ini'), 'w') as f:
        f.write(f'[Sequence]\nname={seq_name}\nseqLength={seq_length}\n')

    seqmap_dir = os.path.join(gt_root, 'seqmaps')
    os.makedirs(seqmap_dir, exist_ok=True)
    with open(os.path.join(seqmap_dir, f'{BENCHMARK}-{SPLIT}.txt'), 'w') as f:
        f.write('name\n' + seq_name + '\n')

    trk_data_dir = os.path.join(trk_root, f'{BENCHMARK}-{SPLIT}', TRACKER, 'data')
    os.makedirs(trk_data_dir, exist_ok=True)
    shutil.copy(output_txt, os.path.join(trk_data_dir, f'{seq_name}.txt'))

    eval_config = trackeval.Evaluator.get_default_eval_config()
    eval_config.update({
        'USE_PARALLEL': False, 'NUM_PARALLEL_CORES': 1,
        'PRINT_RESULTS': False, 'PRINT_ONLY_COMBINED': False,
        'PRINT_CONFIG': False, 'TIME_PROGRESS': False,
        'OUTPUT_SUMMARY': False, 'OUTPUT_EMPTY_CLASSES': False,
        'OUTPUT_DETAILED': False, 'PLOT_CURVES': False,
    })
    dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    dataset_config.update({
        'GT_FOLDER': gt_root, 'TRACKERS_FOLDER': trk_root, 'OUTPUT_FOLDER': work,
        'BENCHMARK': BENCHMARK, 'SPLIT_TO_EVAL': SPLIT,
        'TRACKERS_TO_EVAL': [TRACKER], 'CLASSES_TO_EVAL': ['pedestrian'],
        'DO_PREPROC': True, 'PRINT_CONFIG': False,
    })
    metrics_config = {'METRICS': ['HOTA'], 'THRESHOLD': 0.5, 'PRINT_CONFIG': False}

    evaluator    = trackeval.Evaluator(eval_config)
    dataset_list = [trackeval.datasets.MotChallenge2DBox(dataset_config)]
    metrics_list = [trackeval.metrics.HOTA(metrics_config)]

    out, _ = evaluator.evaluate(dataset_list, metrics_list)
    hres = out['MotChallenge2DBox'][TRACKER][seq_name]['pedestrian']['HOTA']

    alphas = np.arange(0.05, 1.0, 0.05)
    per_alpha = pd.DataFrame({
        'alpha': alphas, 'HOTA': hres['HOTA'], 'DetA': hres['DetA'],
        'AssA': hres['AssA'], 'DetRe': hres['DetRe'], 'DetPr': hres['DetPr'],
        'AssRe': hres['AssRe'], 'AssPr': hres['AssPr'], 'LocA': hres['LocA'],
    })
    hota_summary = {k: float(np.mean(hres[k])) for k in
                    ('HOTA','DetA','AssA','DetRe','DetPr','AssRe','AssPr','LocA')}

    pd.DataFrame([hota_summary]).to_csv(summary_csv, index=False)
    per_alpha.to_csv(per_alpha_csv, index=False)
    shutil.rmtree(work, ignore_errors=True)

    return {k: v * 100 for k, v in hota_summary.items()}


print('✅ Helper functions loaded: load_sequence, run_tracking, eval_mota, eval_hota')


In [ ]:
# ============================================================================
# CELL 5: Driver — loop over all sequences, save per-seq + combined metrics
# ============================================================================
# For each sequence in SEQUENCES: track → score MOTA → score HOTA, append a
# row to the combined table, and write ALL_SEQUENCES_metrics.csv after EVERY
# sequence (so a crash mid-batch still leaves you the rows completed so far).

import os, traceback, time
import pandas as pd
from google.colab import drive as _drive

def ensure_drive(test_path=None):
    test = test_path or OUTPUT_DIR
    for attempt in range(3):
        if os.path.isdir(test): return
        print(f'   Drive disconnected (attempt {attempt+1}/3) — remounting...')
        try: _drive.flush_and_unmount()
        except Exception: pass
        _drive.mount('${DRIVE_ROOT}', force_remount=True)
        time.sleep(8)
    if not os.path.isdir(test):
        raise RuntimeError('Drive unreachable after 3 remount attempts.')

ensure_drive()
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Drive OK. Output: {OUTPUT_DIR}')

all_rows = []

# If a combined CSV already exists and we're resuming, load it so we don't
# lose previously-computed rows.
if SKIP_IF_DONE and os.path.exists(COMBINED_CSV):
    try:
        prev = pd.read_csv(COMBINED_CSV)
        # Drop summary rows (MEAN-*, AVERAGE-*) — only real sequences count as done.
        prev = prev[~prev['sequence'].astype(str).str.startswith(('MEAN-', 'AVERAGE'))]
        all_rows = prev.to_dict('records')
        done_already = {r['sequence'] for r in all_rows}
        print(f'Resuming: {len(done_already)} sequence(s) already in {os.path.basename(COMBINED_CSV)}')
    except Exception:
        all_rows, done_already = [], set()
else:
    done_already = set()

# --- Load FROZEN rule + per-sequence track_thresh resolver ------------------
import json as _json, glob as _glob, numpy as _np
with open(RULE_JSON) as _f:
    _RULE = _json.load(_f)
_LO  = _RULE['density_tertiles']['low_below']
_HI  = _RULE['density_tertiles']['high_atabove']
_BTH = _RULE['bucket_track_thresh']
print('Loaded frozen rule from:', RULE_JSON)
print('  bucket_track_thresh:', _BTH)
print(f'  density tertiles: low<{_LO}  high>={_HI} det/frame')

_test_density = {}
if os.path.exists(PERSEQ_DENSITY_CACHE):
    _test_density = _json.load(open(PERSEQ_DENSITY_CACHE))

def _measure_density(seq):
    """Mean detections/frame from OUR detector over up to 60 sampled frames. NO GT."""
    base = seq['seq_name'].rsplit('-',1)[0]
    if base in _test_density: return _test_density[base]
    files = sorted(_glob.glob(os.path.join(seq['frames_dir'], f'*{seq["im_ext"]}')))
    idx = _np.linspace(0, len(files)-1, min(60, len(files))).astype(int)
    counts=[]
    for k in idx:
        det = yolo_detect_batch([cv2.imread(files[k])])[0]
        counts.append(0 if det is None else len(det))
    _test_density[base]=float(_np.mean(counts))
    _json.dump(_test_density, open(PERSEQ_DENSITY_CACHE,'w'), indent=1)
    return _test_density[base]

def _thresh_for(seq):
    d=_measure_density(seq)
    b = 'low' if d < _LO else ('high' if d >= _HI else 'med')
    th=float(_BTH.get(b, 0.5))
    print(f'   density={d:.1f} -> bucket={b} -> track_thresh={th:.2f}')
    return th

for i, seq_name in enumerate(SEQUENCES, 1):
    print(f'\n{"="*72}\n[{i}/{len(SEQUENCES)}]  {seq_name}\n{"="*72}')

    output_txt    = os.path.join(OUTPUT_DIR, f'{seq_name}.txt')
    output_mp4    = os.path.join(OUTPUT_DIR, f'{seq_name}.mp4')
    mota_csv      = os.path.splitext(output_txt)[0] + '_mota_summary.csv'
    hota_csv      = os.path.splitext(output_txt)[0] + '_hota_summary.csv'
    hota_alpha    = os.path.splitext(output_txt)[0] + '_hota_per_alpha.csv'

    # Resume logic: skip if we already have this sequence in the combined table
    # AND its output .txt exists.
    if SKIP_IF_DONE and os.path.exists(output_txt) and os.path.getsize(output_txt) > 0:
        print('   already done — skipping (set SKIP_IF_DONE=False to force).')
        continue

    ensure_drive(os.path.join(MOT17_TRAIN_DIR, seq_name))
    try:
        seq = load_sequence(seq_name)
    except Exception as e:
        print(f'   x load failed: {e}')
        traceback.print_exc()
        continue

    try:
        # 1) Track + write MOT-format .txt. TEST set has NO ground truth, so we
        #    DO NOT evaluate -- we only produce the result file for submission.
        if (SKIP_IF_DONE and os.path.exists(output_txt)
                and os.path.getsize(output_txt) > 0):
            print('   tracker output exists — reusing it.')
        else:
            globals()['TRACK_THRESH'] = _thresh_for(seq)   # frozen-rule per-seq thresh
            run_tracking(seq, output_txt,
                         render_mp4=RENDER_MP4, output_mp4=output_mp4)
        print(f'   wrote result -> {os.path.basename(output_txt)}')
        # NO eval_mota / eval_hota on TEST (no GT). Skip scoring entirely.

        # (TEST: no scoring table — submission export only)

    except Exception:
        print(f'   ✗ ERROR on {seq_name}:')
        traceback.print_exc()
        print('   continuing to next sequence...')
        continue

# ---- Final combined table + MOT-style averages ----------------------------
# MOT17 reporting convention: a tracker is scored on all 21 sequence-detector
# combinations; the headline number is the unweighted mean over those 21. We
# also report the per-detector means (DPM / FRCNN / SDP) so reviewers can see
# robustness across detection quality, which is the whole point of the protocol.
print(f'\n{"="*72}\nDONE — combined results\n{"="*72}')
# TEST set: no scoring table is produced (no GT). Per-sequence result .txt
# files are written to OUTPUT_DIR. Run the next cell to package them.
print(f'\nAll sequences tracked. Result .txt files are in:\n  {OUTPUT_DIR}')
print('Now run the final cell to zip them for MOTChallenge submission.')


In [ ]:
# ============================================================================
# PACKAGE FOR MOTCHALLENGE SUBMISSION
# Zip the 21 result .txt files FLAT (no subfolders), named MOT17-XX-DET.txt.
# Upload this zip at motchallenge.net -> MOT17 -> Submit (declare: PRIVATE det).
# ============================================================================
import os, zipfile, glob

SUBMIT_ZIP = os.path.join(os.path.dirname(OUTPUT_DIR), 'DAAT_MOT17_test_PERSEQ_submission.zip')
txts = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'MOT17-*.txt')))
# keep only the 21 result files (exclude any *_registry.csv etc.)
txts = [t for t in txts if os.path.basename(t).count('-') == 2
        and os.path.basename(t).split('-')[-1].split('.')[0] in ('DPM','FRCNN','SDP')]

print(f'Found {len(txts)} result files:')
for t in txts: print('  ', os.path.basename(t))
assert len(txts) == 21, f'Expected 21 result files, found {len(txts)}. Run all sequences first.'

with zipfile.ZipFile(SUBMIT_ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for t in txts:
        z.write(t, arcname=os.path.basename(t))   # FLAT: filename only, no folder

print(f'\n✅ Submission zip written:\n   {SUBMIT_ZIP}')
print('   Upload at https://motchallenge.net  ->  MOT17  ->  Submit your results')
print('   Declare detections as: PRIVATE (YOLOX-X).')
